In [1]:
pip install pyvi

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 23.2 MB/s eta 0:00:0000:0100:01
  Obtaining dependency information for python-crfsuite>=0.8.3 from https://files.pythonhosted.org/packages/38/1d/c475ba7d11e9735f00eb08e2f5315aa2e21c24cc85a0474c3fd425edef58/python_crfsuite-0.9.10-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 58.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install gensim

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install vncorenlp

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 9.8 MB/s eta 0:00:00:00:010:01
  Preparing metadata (setup.py) ... done
  Created wheel for vncorenlp: filename=vncorenlp-1.0.3-py3-none-any.whl size=2645932 sha256=9bd1ec1c34e5f760cdf00c1a8f2b10053d3ed349ebc344be171d45a48b2e8496
  Stored in directory: /root/.cache/pip/wheels/5d/d9/b3/41f6c6b1ab758561fd4aab55dc0480b9d7a131c6aaa573a3fa
Successfully built vncorenlp
Note: you may need to restart the kernel to use updated packages.


In [4]:
!git clone https://github.com/vncorenlp/VnCoreNLP.git

Cloning into 'VnCoreNLP'...
remote: Enumerating objects: 259, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 259 (delta 17), reused 14 (delta 2), pack-reused 212
Receiving objects: 100% (259/259), 237.79 MiB | 50.67 MiB/s, done.
Resolving deltas: 100% (93/93), done.


In [5]:
!rm -rf teencode.txt
!wget https://gist.githubusercontent.com/nguyenvanhieuvn/7d9441c10b3c2739499fc5a4d9ea06fb/raw/df939245b3e841b62af115be4dcb3516dadc9fc5/teencode.txt

--2023-12-21 01:59:48--  https://gist.githubusercontent.com/nguyenvanhieuvn/7d9441c10b3c2739499fc5a4d9ea06fb/raw/df939245b3e841b62af115be4dcb3516dadc9fc5/teencode.txt
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.111.133, 185.199.110.133, 185.199.109.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5656 (5.5K) [text/plain]
Saving to: 'teencode.txt'

teencode.txt        100%[===================>]   5.52K  --.-KB/s    in 0s      

2023-12-21 01:59:48 (57.6 MB/s) - 'teencode.txt' saved [5656/5656]



In [6]:
import sys
sys.path.insert(1, '/kaggle/input/preprocess')

from preprocess import (
    remove_HTML,
    convert_unicode,
    # standardize_sentence_typing,
    normalize_acronyms,
    #word_segmentation, # When use PhoBERT
    remove_unnecessary_characters
)

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.24.3
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [7]:
import pandas as pd
import numpy as np
import re
from pyvi import ViTokenizer
from sklearn.model_selection import train_test_split
import tensorflow as tf
from keras.layers import Dense, Dropout, Concatenate, Input
from keras.layers import LSTM, Embedding, Bidirectional, GRU
from keras.layers import SpatialDropout1D, Conv1D, GlobalAveragePooling1D, GlobalMaxPooling1D
from keras.preprocessing.text import Tokenizer
import seaborn as sns
from gensim import models
from keras.initializers import Constant
from keras.preprocessing.sequence import pad_sequences
from keras.losses import BinaryCrossentropy, CategoricalCrossentropy
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

# I. Read Data

In [8]:
df_train = pd.read_csv('/kaggle/input/sneaker-reviews-2-0/train_data_final.csv')
df_val = pd.read_csv('/kaggle/input/sneaker-reviews-2-0/val_data_final.csv')
df_test = pd.read_csv('/kaggle/input/sneaker-reviews-2-0/test_data_final.csv')

In [9]:
df_train = df_train.rename(columns={'Review': 'Comment'})
df_val = df_val.rename(columns={'Review': 'Comment'})
df_test = df_test.rename(columns={'Review': 'Comment'})

In [10]:
df = pd.read_csv('/kaggle/input/sneaker-reviews/Raw data.csv')

In [11]:
df_train['Others'] = [-1] * len(df_train)
df_test['Others'] = [-1] * len(df_test)
df_val['Others'] = [-1] * len(df_val)

In [12]:
def convert_others(dft, df):
    for i in range(len(dft)):
        idx = df[df['Comment'] == dft['Comment'][i]].index.values[0]
        if type(df['Others'][idx]) == str:
            if(df['Others'][idx] == 'x'):
                dft.loc[i, 'Others'] = 2
            else:
                dft.loc[i, 'Others'] = int(df['Others'][idx])
    return dft

In [13]:
df_train = convert_others(df_train, df)
df_test = convert_others(df_test, df)
df_val = convert_others(df_val, df)

# II. Preprocess

In [14]:
import regex as re
import string
import emoji

from vncorenlp import VnCoreNLP
from nltk import flatten

In [15]:
def text_preprocess(text):
    text = remove_HTML(text)
    text = convert_unicode(text)
    # text = standardize_sentence_typing(text)
    text = normalize_acronyms(text)
    #text = word_segmentation(text) # When use PhoBERT
    text = remove_unnecessary_characters(text)
    return text.lower()
    #return text

In [16]:
WORDS_NO_UNIKEY = {
    "ă": "aw", "ắ": "aws", "ằ": "awf", "ẳ": "awr", "ẵ": "awx", "ặ": "awj",
    "â": "aa", "ấ": "aas", "ậ": "aaj", "ẩ": "aar", "ầ": "aaf", "ẫ": "aax",
    "á": "as", "à": "af", "ả": "ar", "ã": "ax", "ạ": "aj",
    "ô": "oo", "ố": "oos", "ồ": "oof", "ổ": "oor", "ỗ": "oox", "ộ": "ooj",
    "ó": "os", "ò": "of", "ỏ": "or", "õ": "ox", "ọ": "oj",
    "ơ": "ow", "ớ": "ows", "ờ": "owf", "ở": "owr", "ỡ": "owx", "ợ": "owj",
    "é": "es", "è": "ef", "ẻ": "er", "ẽ": "ex", "ẹ": "ej",
    "ê": "ee", "ế": "ees", "ề": "eef", "ể": "eer", "ễ": "eex", "ệ": "eej",
    "ú": "us", "ù": "uf", "ủ": "ur", "ũ": "ux", "ụ": "uj",
    "ư": "uw", "ứ": "uws", "ừ": "uwf", "ử": "uwr", "ữ": "uwx", "ự": "uwj",
    "í": "is", "ì": "if", "ỉ": "ir", "ị": "ij", "ĩ": "ix",
    "ý": "ys", "ỳ": "yf", "ỷ": "yr", "ỵ": "yj", "ỹ": "yx", "đ": "dd",
}

In [17]:
def remove_duplication(word):
    if word == '':
        return word
    prev_word = ""
    output = ""
    for c in word:
        if c != prev_word:
            _c = c.lower()
            raw_word = WORDS_NO_UNIKEY.get(_c, _c)[0]
            if prev_word == "" or raw_word != WORDS_NO_UNIKEY.get(prev_word, prev_word)[0]:
                output += c
                prev_word = raw_word
    return output

In [18]:
def preprocess(cm):
    new_cm = text_preprocess(cm)
    new_cm = remove_duplication(new_cm)
    new_cm = ViTokenizer.tokenize(new_cm)
    return new_cm

In [19]:
df_train['Preprocessed'] = df_train['Comment'].apply(preprocess)
df_val['Preprocessed'] = df_val['Comment'].apply(preprocess)
df_test['Preprocessed'] = df_test['Comment'].apply(preprocess)

In [20]:
cols = df_train.columns.drop(['Comment', 'Preprocessed'])
for col in cols:
    df_train[col] = df_train[col].astype(int)
    df_val[col] = df_val[col].astype(int)
    df_test[col] = df_test[col].astype(int)

In [21]:
df_train

,Comment,Price,Shipping,Outlook,Quality,Size,Shop_Service,General,Others,Preprocessed
0,"Giày đẹp, đi êm lắm",-1,-1,1,1,-1,-1,-1,-1,giày đẹp đi êm lắm
1,Mình săn sale với giá khá rẻ Chất lượng ok Sh...,1,1,-1,1,-1,-1,-1,-1,mình săn sale với giá khá rẻ chất_lượng ok shi...
2,Hình ảnh và video chỉ mang tính chất minh họa ...,1,1,-1,-1,-1,-1,1,-1,hình_ảnh và video chỉ mang tính_chất minh_họa ...
3,Mình đặt size 39 nhưng chật k đeo nổi. Còn giầ...,-1,1,1,-1,0,-1,-1,-1,mình đặt cỡ 39 nhưng chật không đeo nổi còn gi...
4,Nên mua nha mọi người đẹp xuất sắc lun ạ,-1,-1,1,-1,-1,-1,-1,-1,nên mua nha mọi người đẹp xuất_sắc lun ạ
...,...,...,...,...,...,...,...,...,...,...
8419,"sản phẩm như mô tả, chất lượng tốt, đáng sử dụ...",-1,-1,-1,1,-1,1,-1,-1,sản_phẩm như mô_tả chất_lượng tốt đáng sử_dụng...
8420,"sản phẩm tốt mọi nguòi nên mua ạ,giá tiền phù ...",2,-1,-1,-1,-1,-1,1,-1,sản_phẩm tốt mọi nguòi nên mua ạ giá tiền phù_...
8421,Nên mua nhé giày đẹp lắm ❤,-1,-1,1,-1,-1,-1,-1,-1,nên mua nhé giày đẹp lắm red_heart
8422,Hàng đẹp lắm ạ. Mk mua 2 đôi đều ưng ạ. Shop u...,-1,-1,1,-1,-1,1,-1,-1,hàng đẹp lắm ạ mình mua 2 đôi đều ưng ạ cửa_hà...


# III. Create Input

In [22]:
def get_label_stage_1(df):
    is_price = df.Price.replace({-1: 0, 0: 1, 2: 1})
    is_shipping = df.Shipping.replace({-1: 0, 0: 1, 2: 1})
    is_outlook = df.Outlook.replace({-1: 0, 0: 1, 2: 1})
    is_quality = df.Quality.replace({-1: 0, 0: 1, 2: 1})
    is_size = df.Size.replace({-1: 0, 0: 1, 2: 1})
    is_shopservice = df.Shop_Service.replace({-1: 0, 0: 1, 2: 1})
    is_general = df.General.replace({-1: 0, 0: 1, 2: 1})
    is_others = df.Others.replace({-1: 0, 0: 1, 2: 1})
    dict = {
          'is_price': is_price, 
          'is_shipping': is_shipping,
          'is_outlook': is_outlook,
          'is_quality': is_quality,
          'is_size': is_size,
          'is_shopservice': is_shopservice,
          'is_general': is_general, 
          'is_others': is_others
    }
    df_aspect = pd.DataFrame(dict)
    return df_aspect

In [23]:
def get_index_label(label_stage_df, aspect):
    return [indx_ for indx_ in label_stage_df[label_stage_df['is_{}'.format(aspect)]==1].index.values if indx_ != 0]

def get_list_index_label(label_stage_df, lst_aspect):
    lst_indx_label = {}
    for aspect in lst_aspect:
        lst_indx_label[aspect] = get_index_label(label_stage_df, aspect)
        
    return lst_indx_label

def get_data_stage_2(lst_indx_label, df):
    data = {}
    for aspect, indx in lst_indx_label.items():
        data[aspect] = df.iloc[indx]
    return data

def get_label_stage_2(df_input):
    df_stage_2 = df_input.copy()

    y_price = tf.keras.utils.to_categorical(df_stage_2['price'].Price, num_classes = 3)
    y_shipping = tf.keras.utils.to_categorical(df_stage_2['shipping'].Shipping, num_classes = 3)
    y_outlook = tf.keras.utils.to_categorical(df_stage_2['outlook'].Outlook, num_classes = 3)
    y_quality = tf.keras.utils.to_categorical(df_stage_2['quality'].Quality, num_classes = 3)
    y_others = df_stage_2['others'].Others.replace({-1:0}).to_numpy()
    y_size = tf.keras.utils.to_categorical(df_stage_2['size'].Size, num_classes = 3)
    y_shopservice = tf.keras.utils.to_categorical(df_stage_2['shopservice'].Shop_Service, num_classes = 3)
    y_general = tf.keras.utils.to_categorical(df_stage_2['general'].General, num_classes = 3)
    
    dict2 = {'price': y_price, 
             'shipping': y_shipping,
             'outlook': y_outlook,
             'quality': y_quality,
             'others': y_others,
             'size': y_size,
             'shopservice': y_shopservice,
             'general': y_general}
    del df_stage_2
    
    return dict2

In [24]:
def get_label_stage_3(df_input):
    df_stage_3 = df_input.copy()

    y_price = tf.keras.utils.to_categorical(df_stage_3['Price'], num_classes = 4)
    y_shipping = tf.keras.utils.to_categorical(df_stage_3['Shipping'], num_classes = 4)
    y_outlook = tf.keras.utils.to_categorical(df_stage_3['Outlook'], num_classes = 4)
    y_quality = tf.keras.utils.to_categorical(df_stage_3['Quality'], num_classes = 4)
    y_others = df_stage_3['Others'].replace({-1:0}).to_numpy()
    y_size = tf.keras.utils.to_categorical(df_stage_3['Size'], num_classes = 4)
    y_shopservice = tf.keras.utils.to_categorical(df_stage_3['Shop_Service'], num_classes = 4)
    y_general = tf.keras.utils.to_categorical(df_stage_3['General'], num_classes = 4)
    
    dict3 = {'price': y_price, 
             'shipping': y_shipping,
             'outlook': y_outlook,
             'quality': y_quality,
             'others': y_others,
             'size': y_size,
             'shopservice': y_shopservice,
             'general': y_general}
    del df_stage_3
    
    return dict3

In [25]:
def get_cmt(dict_df, lst_aspect):
    dict_cmt = {}
    for aspect in lst_aspect:
        dict_cmt[aspect] = dict_df[aspect]['Preprocessed'].values
    return dict_cmt

In [26]:
def get_array_label(df):
    label_stage_1_df = get_label_stage_1(df)
    lst_aspect = ['price', 'shipping', 'outlook', 'quality', 'size', 'shopservice', 'general', 'others']
    lst_indx = get_list_index_label(label_stage_1_df, lst_aspect)
    
    df_temp = get_data_stage_2(lst_indx, df)
    cmt_df = get_cmt(df_temp, lst_aspect)
    
    label_stage_2 = get_label_stage_2(df_temp)
    label_stage_3 = get_label_stage_3(df)
    del label_stage_1_df, lst_aspect, df_temp
    
    return cmt_df, label_stage_2, label_stage_3

In [27]:
df_train.shape, df_val.shape, df_test.shape

((8424, 10), (936, 10), (2340, 10))

In [28]:
label_aspect_train = get_label_stage_1(df_train)
label_aspect_val = get_label_stage_1(df_val)
label_aspect_test = get_label_stage_1(df_test)

In [29]:
label_aspect_train

,is_price,is_shipping,is_outlook,is_quality,is_size,is_shopservice,is_general,is_others
0,0,0,1,1,0,0,0,0
1,1,1,0,1,0,0,0,0
2,1,1,0,0,0,0,1,0
3,0,1,1,0,1,0,0,0
4,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...
8419,0,0,0,1,0,1,0,0
8420,1,0,0,0,0,0,1,0
8421,0,0,1,0,0,0,0,0
8422,0,0,1,0,0,1,0,0


In [30]:
cmt_train, label_sentiment_train, lb_train = get_array_label(df_train)
cmt_val, label_sentiment_val, lb_val = get_array_label(df_val)
cmt_test, label_sentiment_test, lb_test = get_array_label(df_test)

# IV. Embedding

In [31]:
max_len = 200

In [32]:
def get_tokenize(train, val, test):
    tok = Tokenizer(filters='')
    tok.fit_on_texts(train)
    tok.fit_on_texts(val)
    tok.fit_on_texts(test)
    
    tokenized_train = tok.texts_to_sequences(train)
    tokenized_val = tok.texts_to_sequences(val)
    tokenized_test = tok.texts_to_sequences(test)
    
    return tok, tokenized_train, tokenized_val, tokenized_test 

In [33]:
def get_padded(max_len, tokenized_train, tokenized_val, tokenized_test):
    padded_train = pad_sequences(tokenized_train, padding = 'post', maxlen = max_len)
    padded_val = pad_sequences(tokenized_val, padding = 'post', maxlen = max_len)
    padded_test = pad_sequences(tokenized_test, padding = 'post', maxlen = max_len)
    
    return padded_train, padded_val, padded_test 

In [34]:
def get_embedding_matrix(embedding_dim, vocab_size, tok):
    embed_matrix = np.zeros(shape=(vocab_size, embedding_dim))
    word_failed = []

    for word, i in tok.word_index.items():
        try:
            embed_vector = w2c_model[word]
        except:
            word_failed.append(word)
        else:
            embed_matrix[i] = embed_vector
            
    return embed_matrix

In [35]:
w2v_path = '/kaggle/input/elmo-sen/elmo_aivivn_train_test_wikn_sen.txt'
w2c_model = models.KeyedVectors.load_word2vec_format(w2v_path, binary = False)

In [36]:
tokenizer_aspect, tokenized_aspect_train, tokenized_aspect_val, tokenized_aspect_test = get_tokenize(df_train['Preprocessed'].values, df_val['Preprocessed'].values, df_test['Preprocessed'].values)

In [37]:
padded_aspect_train, padded_aspect_val, padded_aspect_test = get_padded(max_len, tokenized_aspect_train, tokenized_aspect_val, tokenized_aspect_test)

In [38]:
embedding_dim = 1024 
vocab_aspect_size = len(tokenizer_aspect.word_index)+1
embedding_aspect_matrix = get_embedding_matrix(embedding_dim, vocab_aspect_size, tokenizer_aspect)

In [39]:
embedding_aspect_matrix.shape

(10034, 1024)

# V. Model

## 1. BiLSTM

## a) Build Model

In [40]:
def bilstm_aspect(vocab_aspect_size, embedding_dim, max_len, embedding_aspect_matrix, padded_aspect_train, padded_aspect_val, label_aspect_train, label_aspect_val, epochs=70):
    input = Input(shape=(max_len,))
    embed = Embedding(input_dim=vocab_aspect_size,
                      output_dim=embedding_dim,
                      embeddings_initializer=Constant(embedding_aspect_matrix),
                      input_length=max_len,
                      trainable=True)(input)
    lstm = Bidirectional(LSTM(units = 200, activation = 'tanh'))(embed)
    
    # aspect 
    aspect_dense2 = Dense(128, activation='relu')(lstm)
    aspect_dropout1 = Dropout(0.2)(aspect_dense2)
    aspect_dense3 = Dense(64, activation='relu')(aspect_dropout1)
    aspect_dense4 = Dense(32, activation='relu')(aspect_dense3)
    aspect_dense5 = Dense(8, activation='sigmoid')(aspect_dense4)
    
    aspect_model = tf.keras.Model(inputs = input, outputs = aspect_dense5)
    aspect_model.compile(optimizer=Adam(learning_rate = 0.0001), loss='binary_crossentropy', metrics=['acc'])
    callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)
    
    # Fit model
    history = aspect_model.fit(x=padded_aspect_train,
                               y=label_aspect_train.to_numpy(),
                               validation_data=(padded_aspect_val, label_aspect_val.to_numpy()),
                               batch_size = 128,
                               epochs=epochs,
                               callbacks = [callback],
                               verbose=1)
    
    return aspect_model

In [41]:
aspect_model_bil = bilstm_aspect(vocab_aspect_size, embedding_dim, max_len, embedding_aspect_matrix, padded_aspect_train, padded_aspect_val, label_aspect_train, label_aspect_val)

Epoch 1/70
66/66 [==============================] - 23s 193ms/step - loss: 0.5722 - acc: 0.3403 - val_loss: 0.5012 - val_acc: 0.3611
Epoch 2/70
66/66 [==============================] - 10s 157ms/step - loss: 0.4533 - acc: 0.4180 - val_loss: 0.3914 - val_acc: 0.4840
Epoch 3/70
66/66 [==============================] - 10s 152ms/step - loss: 0.3639 - acc: 0.5134 - val_loss: 0.3184 - val_acc: 0.5673
Epoch 4/70
66/66 [==============================] - 9s 137ms/step - loss: 0.3082 - acc: 0.5643 - val_loss: 0.2757 - val_acc: 0.5897
Epoch 5/70
66/66 [==============================] - 8s 127ms/step - loss: 0.2656 - acc: 0.5946 - val_loss: 0.2514 - val_acc: 0.5994
Epoch 6/70
66/66 [==============================] - 9s 132ms/step - loss: 0.2400 - acc: 0.6064 - val_loss: 0.2387 - val_acc: 0.5919
Epoch 7/70
66/66 [==============================] - 8s 127ms/step - loss: 0.2187 - acc: 0.6269 - val_loss: 0.2316 - val_acc: 0.5908
Epoch 8/70
66/66 [==============================] - 8s 117ms/step - loss:

In [42]:
pred_train = aspect_model_bil.predict(padded_aspect_train)
pred_val = aspect_model_bil.predict(padded_aspect_val)
pred_test = aspect_model_bil.predict(padded_aspect_test)

df_train_pred = round(pd.DataFrame(pred_train), 0)
df_train_true_ = label_aspect_train
df_train_pred.columns = df_train_true_.columns

df_val_pred = round(pd.DataFrame(pred_val), 0)
df_val_true_ = label_aspect_val
df_val_pred.columns = df_val_true_.columns

df_test_pred = round(pd.DataFrame(pred_test), 0)
df_test_true_ = label_aspect_test
df_test_pred.columns = df_test_true_.columns

74/74 [==============================] - 1s 13ms/step


In [43]:
#import os, glob
#for filename in glob.glob("/kaggle/working/elmo*"):
    #os.remove(filename) 

In [44]:
def get_pd_report(label_aspect_train, df_test_true_, df_test_pred):
    aspect_result_bilstm_report = {}
    for col in label_aspect_train.columns:
        aspect_result_bilstm = classification_report(df_test_true_[col], df_test_pred[col], output_dict=True)
        aspect_result_bilstm_report[col] = pd.DataFrame(aspect_result_bilstm).T
        aspect_result_bilstm_report[col]['aspect'] = col
        
    output_aspect_report = pd.DataFrame()
    for indx, val in aspect_result_bilstm_report.items():
        output_aspect_report = pd.concat([output_aspect_report, aspect_result_bilstm_report[indx]])
    
    return output_aspect_report 

In [45]:
aspect_result_bilstm_report = {}
for col in label_aspect_train.columns:
    print(col)
    aspect_result_bilstm = classification_report(df_test_true_[col], df_test_pred[col], output_dict=True)
    print(aspect_result_bilstm)
    
    aspect_result_bilstm_report[col] = pd.DataFrame(aspect_result_bilstm).T
    aspect_result_bilstm_report[col]['aspect'] = col
    
    output_aspect_report = pd.DataFrame()
    for indx, val in aspect_result_bilstm_report.items():
        output_aspect_report = pd.concat([output_aspect_report, aspect_result_bilstm_report[indx]])

is_price
{'0': {'precision': 0.9499518768046198, 'recall': 0.9874937468734367, 'f1-score': 0.9683590875643855, 'support': 1999}, '1': {'precision': 0.9045801526717557, 'recall': 0.6950146627565983, 'f1-score': 0.7860696517412937, 'support': 341}, 'accuracy': 0.9448717948717948, 'macro avg': {'precision': 0.9272660147381877, 'recall': 0.8412542048150176, 'f1-score': 0.8772143696528396, 'support': 2340}, 'weighted avg': {'precision': 0.9433400144416683, 'recall': 0.9448717948717948, 'f1-score': 0.9417946868739263, 'support': 2340}}
is_shipping
{'0': {'precision': 0.9725609756097561, 'recall': 0.9755351681957186, 'f1-score': 0.9740458015267176, 'support': 1635}, '1': {'precision': 0.9428571428571428, 'recall': 0.9361702127659575, 'f1-score': 0.9395017793594306, 'support': 705}, 'accuracy': 0.9636752136752137, 'macro avg': {'precision': 0.9577090592334494, 'recall': 0.955852690480838, 'f1-score': 0.9567737904430741, 'support': 2340}, 'weighted avg': {'precision': 0.9636117439471098, 'recal

In [46]:
for col in label_aspect_train.columns:
    print(col)
    aspect_result = classification_report(df_test_true_[col], df_test_pred[col])

    print(aspect_result)

is_price
              precision    recall  f1-score   support

           0       0.95      0.99      0.97      1999
           1       0.90      0.70      0.79       341

    accuracy                           0.94      2340
   macro avg       0.93      0.84      0.88      2340
weighted avg       0.94      0.94      0.94      2340

is_shipping
              precision    recall  f1-score   support

           0       0.97      0.98      0.97      1635
           1       0.94      0.94      0.94       705

    accuracy                           0.96      2340
   macro avg       0.96      0.96      0.96      2340
weighted avg       0.96      0.96      0.96      2340

is_outlook
              precision    recall  f1-score   support

           0       0.88      0.93      0.90      1069
           1       0.94      0.90      0.92      1271

    accuracy                           0.91      2340
   macro avg       0.91      0.91      0.91      2340
weighted avg       0.91      0.91      0.9

In [47]:
path = '/kaggle/working/'

In [48]:
def save_result_pred_aspect(pred_aspect, true_aspect, data_eval, model_type):
    df_pred = round(pd.DataFrame(pred_aspect), 0)
    df_true = pd.DataFrame(true_aspect)
    for i, aspect in enumerate(['price', 'shipping', 'outlook', 'quality',
                              'size', 'shopservice', 'general', 'others']):
        if data_eval == 'test':
            with open(f'{path}elmo_{model_type}_aspect_test_result.txt', "a") as dest:
                dest.write("Classification report for aspect: {} \n".format(aspect.upper()))
                dest.write(classification_report(df_true[f'is_{aspect}'], df_pred[f'is_{aspect}']))
    
        elif data_eval == 'eval':
            with open(f'{path}elmo_{model_type}_aspect_val_result.txt', "a") as dest:
                dest.write("Classification report for aspect: {} \n".format(aspect.upper()))
                dest.write(classification_report(df_true[f'is_{aspect}'], df_pred[f'is_{aspect}']))
        else:
            raise('data eval is invalid')

In [49]:
save_result_pred_aspect(df_test_true_, df_test_pred, data_eval = 'test', model_type = 'bilstm')
save_result_pred_aspect(df_val_true_, df_val_pred, data_eval = 'eval', model_type = 'bilstm')

In [50]:
def bilstm_polarity(aspect, vocab_sentiment_size, embedding_dim, max_len, embedding_sentiment_matrix, padded_sentiment_train, padded_sentiment_val, label_sentiment_train, label_sentiment_val, epochs=70):
    input = Input(shape=(max_len,))
    embed = Embedding(input_dim=vocab_sentiment_size[aspect],
                  output_dim=embedding_dim,
                  embeddings_initializer=Constant(embedding_sentiment_matrix[aspect]),
                  input_length=max_len,
                  trainable=True)(input)
    lstm = Bidirectional(LSTM(units = 200, activation = 'tanh'))(embed)
    sentiment_dense2 = Dense(128, activation='relu')(lstm)
    sentiment_dropout1 = Dropout(0.2)(sentiment_dense2)
    sentiment_dense3 = Dense(64, activation='relu')(sentiment_dropout1)
    sentiment_dense4 = Dense(32, activation='relu')(sentiment_dense3)
    out_sentiment = Dense(units = 3, activation = 'softmax')(sentiment_dense4)
    
    sentiment_model_bil[aspect] = tf.keras.Model(inputs = input, outputs = out_sentiment)
    sentiment_model_bil[aspect].compile(optimizer=Adam(learning_rate = 0.0001), loss='binary_crossentropy', metrics=['acc'])
    callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)
    
    history = sentiment_model_bil[aspect].fit(x=padded_sentiment_train[aspect],
                                          y=label_sentiment_train[aspect],
                                          validation_data=(padded_sentiment_val[aspect], label_sentiment_val[aspect]),
                                          batch_size = 128,
                                          epochs=epochs,
                                          callbacks = [callback],
                                          verbose=1)
    
    return sentiment_model_bil[aspect]

In [51]:
lst_aspect = []
for col in label_aspect_train.columns.values:
    if col!='is_others':
        lst_aspect.append(col.split('_')[1])

In [52]:
tokenized_sentiment_train = {}
tokenized_sentiment_val = {}
tokenized_sentiment_test = {}
tokenizer_sentiment = {}

for aspect in lst_aspect:
    tokenizer_sentiment[aspect], tokenized_sentiment_train[aspect], tokenized_sentiment_val[aspect], tokenized_sentiment_test[aspect] = get_tokenize(cmt_train[aspect], cmt_val[aspect], cmt_test[aspect])

padded_sentiment_train = {}
padded_sentiment_val = {}
padded_sentiment_test = {}
for aspect in lst_aspect:
    padded_sentiment_train[aspect], padded_sentiment_val[aspect], padded_sentiment_test[aspect] = get_padded(max_len, tokenized_sentiment_train[aspect], tokenized_sentiment_val[aspect], tokenized_sentiment_test[aspect])

#embedding matrix stage 2
vocab_sentiment_size = {}
embedding_sentiment_matrix = {}
for aspect in lst_aspect:
    vocab_sentiment_size[aspect]  = len(tokenizer_sentiment[aspect].word_index)+1
    embedding_sentiment_matrix[aspect] = get_embedding_matrix(embedding_dim, vocab_sentiment_size[aspect], tokenizer_sentiment[aspect])

In [53]:
sentiment_model_bil = {}
for aspect in lst_aspect:
    sentiment_model_bil[aspect] = bilstm_polarity(aspect, vocab_sentiment_size, embedding_dim, max_len, embedding_sentiment_matrix, padded_sentiment_train, padded_sentiment_val, label_sentiment_train, label_sentiment_val)

Epoch 1/70
10/10 [==============================] - 7s 280ms/step - loss: 0.6372 - acc: 0.5532 - val_loss: 0.5621 - val_acc: 0.7681
Epoch 2/70
10/10 [==============================] - 2s 162ms/step - loss: 0.5165 - acc: 0.7426 - val_loss: 0.4551 - val_acc: 0.7681
Epoch 3/70
10/10 [==============================] - 2s 172ms/step - loss: 0.4331 - acc: 0.7426 - val_loss: 0.3916 - val_acc: 0.7681
Epoch 4/70
10/10 [==============================] - 2s 187ms/step - loss: 0.3952 - acc: 0.7426 - val_loss: 0.3706 - val_acc: 0.7681
Epoch 5/70
10/10 [==============================] - 2s 163ms/step - loss: 0.3880 - acc: 0.7426 - val_loss: 0.3645 - val_acc: 0.7681
Epoch 6/70
10/10 [==============================] - 2s 176ms/step - loss: 0.3683 - acc: 0.7426 - val_loss: 0.3503 - val_acc: 0.7681
Epoch 7/70
10/10 [==============================] - 2s 174ms/step - loss: 0.3449 - acc: 0.7482 - val_loss: 0.3253 - val_acc: 0.7681
Epoch 8/70
10/10 [==============================] - 1s 150ms/step - loss: 0.

In [54]:
df_sentiment_train_pred = {}
df_sentiment_train_true_ = {}

df_sentiment_val_true_ = {}
df_sentiment_val_pred = {}

df_sentiment_test_true_ = {}
df_sentiment_test_pred = {}


for aspect in lst_aspect:
    pred_sentiment_train = sentiment_model_bil[aspect].predict(padded_sentiment_train[aspect])
    pred_sentiment_val = sentiment_model_bil[aspect].predict(padded_sentiment_val[aspect])
    pred_sentiment_test = sentiment_model_bil[aspect].predict(padded_sentiment_test[aspect])
    
    df_sentiment_train_pred[aspect] = np.argmax(pred_sentiment_train, axis=1)
    df_sentiment_train_true_[aspect] = np.argmax(label_sentiment_train[aspect], axis = 1)
    
    df_sentiment_val_pred[aspect] = np.argmax(pred_sentiment_val, axis=1)
    df_sentiment_val_true_[aspect] = np.argmax(label_sentiment_val[aspect], axis = 1)
    
    df_sentiment_test_pred[aspect] = np.argmax(pred_sentiment_test, axis=1)
    df_sentiment_test_true_[aspect] = np.argmax(label_sentiment_test[aspect], axis=1)

15/15 [==============================] - 0s 13ms/step


In [55]:
def get_pd_report_sentiment(df_sentiment_test_true_, df_sentiment_test_pred):
    sentiment_result_bilstm = {}
    for aspect in lst_aspect:
        sentiment_result_bilstm[aspect] = pd.DataFrame(classification_report(df_sentiment_test_true_[aspect].tolist(), df_sentiment_test_pred[aspect].tolist(), output_dict=True)).T
        sentiment_result_bilstm[aspect]['aspect'] = aspect

    output_sentiment_report = pd.DataFrame()
    for indx, val in sentiment_result_bilstm.items():
        output_sentiment_report = pd.concat([output_sentiment_report, sentiment_result_bilstm[indx]])

    return output_aspect_report

In [56]:
sentiment_result_bilstm = {}
# for aspect in lst_aspect:
for aspect in lst_aspect:
    sentiment_result_bilstm[aspect] = pd.DataFrame(classification_report(df_sentiment_test_true_[aspect].tolist(), df_sentiment_test_pred[aspect].tolist(), output_dict=True)).T
    sentiment_result_bilstm[aspect]['aspect'] = aspect

output_sentiment_report = pd.DataFrame()
for indx, val in sentiment_result_bilstm.items():
    output_sentiment_report = pd.concat([output_sentiment_report, sentiment_result_bilstm[indx]])

/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classif

In [57]:
output_sentiment_report

,precision,recall,f1-score,support,aspect
0,0.000000,0.000000,0.000000,3.000000,price
1,0.944206,0.890688,0.916667,247.000000,price
2,0.731481,0.868132,0.793970,91.000000,price
accuracy,0.876833,0.876833,0.876833,0.876833,price
macro avg,0.558562,0.586273,0.570212,341.000000,price
weighted avg,0.879131,0.876833,0.875859,341.000000,price
0,0.834586,0.895161,0.863813,124.000000,shipping
1,0.961818,0.963570,0.962693,549.000000,shipping
2,0.090909,0.062500,0.074074,32.000000,shipping
accuracy,0.910638,0.910638,0.910638,0.910638,shipping


In [58]:
for aspect in lst_aspect:
    print(aspect)
    print(classification_report(df_sentiment_test_true_[aspect].tolist(), df_sentiment_test_pred[aspect].tolist()))

price
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.94      0.89      0.92       247
           2       0.73      0.87      0.79        91

    accuracy                           0.88       341
   macro avg       0.56      0.59      0.57       341
weighted avg       0.88      0.88      0.88       341

shipping
              precision    recall  f1-score   support

           0       0.83      0.90      0.86       124
           1       0.96      0.96      0.96       549
           2       0.09      0.06      0.07        32

    accuracy                           0.91       705
   macro avg       0.63      0.64      0.63       705
weighted avg       0.90      0.91      0.90       705

outlook
              precision    recall  f1-score   support

           0       0.76      0.56      0.64        95
           1       0.94      0.99      0.96      1118
           2       0.44      0.19      0.27        5

/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classif

In [59]:
for i in lst_aspect:
    with open(f'{path}elmo_bilstm_test_result.txt', "a") as dest:
        dest.write("Classification report for aspect: {} \n".format(i.upper()))
        dest.write(classification_report(df_sentiment_test_true_[i].tolist(), df_sentiment_test_pred[i].tolist()))

/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classif

In [60]:
for i in lst_aspect:
    with open(f'{path}elmo_bilstm_val_result.txt', "a") as dest:
        dest.write("Classification report for aspect: {} \n".format(i.upper()))
        dest.write(classification_report(df_sentiment_val_true_[i].tolist(), df_sentiment_val_pred[i].tolist()))

/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classif

## 2. BiGRU

In [61]:
def gru_aspect(vocab_aspect_size, embedding_dim, max_len, embedding_aspect_matrix, padded_aspect_train, padded_aspect_val, label_aspect_train, label_aspect_val, epochs=70):
    input = Input(shape=(max_len,))
    embed = Embedding(input_dim=vocab_aspect_size,
                      output_dim=embedding_dim,
                      embeddings_initializer=Constant(embedding_aspect_matrix),
                      input_length=max_len,
                      trainable=True)(input)
    lstm = Bidirectional(GRU(units = 200, activation = 'tanh'))(embed)
    
    # aspect 
    aspect_dense2 = Dense(128, activation='relu')(lstm)
    aspect_dropout1 = Dropout(0.2)(aspect_dense2)
    aspect_dense3 = Dense(64, activation='relu')(aspect_dropout1)
    aspect_dense4 = Dense(32, activation='relu')(aspect_dense3)
    aspect_dense5 = Dense(8, activation='sigmoid')(aspect_dense4)
    
    aspect_model = tf.keras.Model(inputs = input, outputs = aspect_dense5)
    aspect_model.compile(optimizer=Adam(learning_rate = 0.0001), loss='binary_crossentropy', metrics=['acc'])
    callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)
    
    # Fit model
    history = aspect_model.fit(x=padded_aspect_train,
                               y=label_aspect_train.to_numpy(),
                               validation_data=(padded_aspect_val, label_aspect_val.to_numpy()),
                               batch_size = 128,
                               epochs=epochs,
                               callbacks = [callback],
                               verbose=1)
    
    return aspect_model

def gru_polarity(aspect, vocab_sentiment_size, embedding_dim, max_len, embedding_sentiment_matrix, padded_sentiment_train, padded_sentiment_val, label_sentiment_train, label_sentiment_val, epochs=70):
    input = Input(shape=(max_len,))
    embed = Embedding(input_dim=vocab_sentiment_size[aspect],
                  output_dim=embedding_dim,
                  embeddings_initializer=Constant(embedding_sentiment_matrix[aspect]),
                  input_length=max_len,
                  trainable=True)(input)
    lstm = Bidirectional(GRU(units = 200, activation = 'tanh'))(embed)
    sentiment_dense2 = Dense(128, activation='relu')(lstm)
    sentiment_dropout1 = Dropout(0.2)(sentiment_dense2)
    sentiment_dense3 = Dense(64, activation='relu')(sentiment_dropout1)
    sentiment_dense4 = Dense(32, activation='relu')(sentiment_dense3)
    out_sentiment = Dense(units = 3, activation = 'softmax')(sentiment_dense4)
    
    sentiment_model_gru[aspect] = tf.keras.Model(inputs = input, outputs = out_sentiment)
    sentiment_model_gru[aspect].compile(optimizer=Adam(learning_rate = 0.0001), loss='binary_crossentropy', metrics=['acc'])
    callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)
    
    history = sentiment_model_gru[aspect].fit(x=padded_sentiment_train[aspect],
                                          y=label_sentiment_train[aspect],
                                          validation_data=(padded_sentiment_val[aspect], label_sentiment_val[aspect]),
                                          batch_size = 128,
                                          epochs=epochs,
                                          callbacks = [callback],
                                          verbose=1)
    
    return sentiment_model_gru[aspect]

In [62]:
aspect_model_gru = gru_aspect(vocab_aspect_size, embedding_dim, max_len, embedding_aspect_matrix, padded_aspect_train, padded_aspect_val, label_aspect_train, label_aspect_val)

Epoch 1/70
66/66 [==============================] - 17s 177ms/step - loss: 0.5966 - acc: 0.2472 - val_loss: 0.5104 - val_acc: 0.3173
Epoch 2/70
66/66 [==============================] - 9s 142ms/step - loss: 0.4859 - acc: 0.3712 - val_loss: 0.4637 - val_acc: 0.4038
Epoch 3/70
66/66 [==============================] - 8s 127ms/step - loss: 0.4486 - acc: 0.4097 - val_loss: 0.4238 - val_acc: 0.4391
Epoch 4/70
66/66 [==============================] - 9s 132ms/step - loss: 0.4083 - acc: 0.4636 - val_loss: 0.3799 - val_acc: 0.4968
Epoch 5/70
66/66 [==============================] - 7s 108ms/step - loss: 0.3707 - acc: 0.4937 - val_loss: 0.3419 - val_acc: 0.5118
Epoch 6/70
66/66 [==============================] - 8s 115ms/step - loss: 0.3337 - acc: 0.5336 - val_loss: 0.3075 - val_acc: 0.5823
Epoch 7/70
66/66 [==============================] - 8s 114ms/step - loss: 0.2984 - acc: 0.5677 - val_loss: 0.2772 - val_acc: 0.5759
Epoch 8/70
66/66 [==============================] - 7s 102ms/step - loss: 0

In [63]:
pred_train = aspect_model_gru.predict(padded_aspect_train)
pred_val = aspect_model_gru.predict(padded_aspect_val)
pred_test = aspect_model_gru.predict(padded_aspect_test)

df_train_pred = round(pd.DataFrame(pred_train), 0)
df_train_true_ = label_aspect_train
df_train_pred.columns = df_train_true_.columns

df_val_pred = round(pd.DataFrame(pred_val), 0)
df_val_true_ = label_aspect_val
df_val_pred.columns = df_val_true_.columns

df_test_pred = round(pd.DataFrame(pred_test), 0)
df_test_true_ = label_aspect_test
df_test_pred.columns = df_test_true_.columns

74/74 [==============================] - 1s 11ms/step


In [64]:
def get_pd_report(label_aspect_train, df_test_true_, df_test_pred):
    aspect_result_gru_report = {}
    for col in label_aspect_train.columns:
        aspect_result_gru_report = classification_report(df_test_true_[col], df_test_pred[col], output_dict=True)
        aspect_result_gru_report[col] = pd.DataFrame(aspect_result_bilstm).T
        aspect_result_gru_report[col]['aspect'] = col
        
    output_aspect_report = pd.DataFrame()
    for indx, val in aspect_result_gru_report.items():
        output_aspect_report = pd.concat([output_aspect_report, aspect_result_gru_report[indx]])
    
    return output_aspect_report 

In [65]:
aspect_result_gru_report = {}
for col in label_aspect_train.columns:
    print(col)
    aspect_result_gru = classification_report(df_test_true_[col], df_test_pred[col], output_dict=True)
    print(aspect_result_gru)
    
    aspect_result_gru_report[col] = pd.DataFrame(aspect_result_gru).T
    aspect_result_gru_report[col]['aspect'] = col
    
    output_aspect_report = pd.DataFrame()
    for indx, val in aspect_result_gru_report.items():
        output_aspect_report = pd.concat([output_aspect_report, aspect_result_gru_report[indx]])

is_price
{'0': {'precision': 0.95947265625, 'recall': 0.9829914957478739, 'f1-score': 0.9710896960711637, 'support': 1999}, '1': {'precision': 0.8835616438356164, 'recall': 0.7565982404692082, 'f1-score': 0.8151658767772513, 'support': 341}, 'accuracy': 0.95, 'macro avg': {'precision': 0.9215171500428082, 'recall': 0.869794868108541, 'f1-score': 0.8931277864242075, 'support': 2340}, 'weighted avg': {'precision': 0.9484104104238013, 'recall': 0.95, 'f1-score': 0.9483674642851706, 'support': 2340}}
is_shipping
{'0': {'precision': 0.977328431372549, 'recall': 0.9755351681957186, 'f1-score': 0.9764309764309763, 'support': 1635}, '1': {'precision': 0.943502824858757, 'recall': 0.9475177304964539, 'f1-score': 0.9455060155697098, 'support': 705}, 'accuracy': 0.9670940170940171, 'macro avg': {'precision': 0.9604156281156531, 'recall': 0.9615264493460862, 'f1-score': 0.9609684960003431, 'support': 2340}, 'weighted avg': {'precision': 0.9671373832562142, 'recall': 0.9670940170940171, 'f1-score':

In [66]:
for col in label_aspect_train.columns:
    print(col)
    aspect_result = classification_report(df_test_true_[col], df_test_pred[col])

    print(aspect_result)

is_price
              precision    recall  f1-score   support

           0       0.96      0.98      0.97      1999
           1       0.88      0.76      0.82       341

    accuracy                           0.95      2340
   macro avg       0.92      0.87      0.89      2340
weighted avg       0.95      0.95      0.95      2340

is_shipping
              precision    recall  f1-score   support

           0       0.98      0.98      0.98      1635
           1       0.94      0.95      0.95       705

    accuracy                           0.97      2340
   macro avg       0.96      0.96      0.96      2340
weighted avg       0.97      0.97      0.97      2340

is_outlook
              precision    recall  f1-score   support

           0       0.91      0.92      0.91      1069
           1       0.93      0.92      0.93      1271

    accuracy                           0.92      2340
   macro avg       0.92      0.92      0.92      2340
weighted avg       0.92      0.92      0.9

In [67]:
path = '/kaggle/working/'

In [68]:
save_result_pred_aspect(df_test_true_, df_test_pred, data_eval = 'test', model_type = 'bigru')
save_result_pred_aspect(df_val_true_, df_val_pred, data_eval = 'eval', model_type = 'bigru')

In [69]:
lst_aspect = []
for col in label_aspect_train.columns.values:
    if col!='is_others':
        lst_aspect.append(col.split('_')[1])

In [70]:
sentiment_model_gru = {}
for aspect in lst_aspect:
    sentiment_model_gru[aspect] = gru_polarity(aspect, vocab_sentiment_size, embedding_dim, max_len, embedding_sentiment_matrix, padded_sentiment_train, padded_sentiment_val, label_sentiment_train, label_sentiment_val)

Epoch 1/70
10/10 [==============================] - 7s 269ms/step - loss: 0.5579 - acc: 0.7290 - val_loss: 0.4813 - val_acc: 0.7681
Epoch 2/70
10/10 [==============================] - 2s 184ms/step - loss: 0.4504 - acc: 0.7426 - val_loss: 0.4120 - val_acc: 0.7681
Epoch 3/70
10/10 [==============================] - 2s 162ms/step - loss: 0.4122 - acc: 0.7426 - val_loss: 0.3881 - val_acc: 0.7681
Epoch 4/70
10/10 [==============================] - 2s 159ms/step - loss: 0.3965 - acc: 0.7426 - val_loss: 0.3712 - val_acc: 0.7681
Epoch 5/70
10/10 [==============================] - 2s 160ms/step - loss: 0.3813 - acc: 0.7426 - val_loss: 0.3618 - val_acc: 0.7681
Epoch 6/70
10/10 [==============================] - 2s 167ms/step - loss: 0.3706 - acc: 0.7426 - val_loss: 0.3549 - val_acc: 0.7681
Epoch 7/70
10/10 [==============================] - 1s 143ms/step - loss: 0.3621 - acc: 0.7426 - val_loss: 0.3483 - val_acc: 0.7681
Epoch 8/70
10/10 [==============================] - 1s 131ms/step - loss: 0.

In [71]:
df_sentiment_train_pred = {}
df_sentiment_train_true_ = {}

df_sentiment_val_true_ = {}
df_sentiment_val_pred = {}

df_sentiment_test_true_ = {}
df_sentiment_test_pred = {}


for aspect in lst_aspect:
    pred_sentiment_train = sentiment_model_gru[aspect].predict(padded_sentiment_train[aspect])
    pred_sentiment_val = sentiment_model_gru[aspect].predict(padded_sentiment_val[aspect])
    pred_sentiment_test = sentiment_model_gru[aspect].predict(padded_sentiment_test[aspect])
    
    df_sentiment_train_pred[aspect] = np.argmax(pred_sentiment_train, axis=1)
    df_sentiment_train_true_[aspect] = np.argmax(label_sentiment_train[aspect], axis = 1)
    
    df_sentiment_val_pred[aspect] = np.argmax(pred_sentiment_val, axis=1)
    df_sentiment_val_true_[aspect] = np.argmax(label_sentiment_val[aspect], axis = 1)
    
    df_sentiment_test_pred[aspect] = np.argmax(pred_sentiment_test, axis=1)
    df_sentiment_test_true_[aspect] = np.argmax(label_sentiment_test[aspect], axis=1)

15/15 [==============================] - 0s 11ms/step


In [72]:
def get_pd_report_sentiment(df_sentiment_test_true_, df_sentiment_test_pred):
    sentiment_result_bilstm = {}
    for aspect in lst_aspect:
        sentiment_result_bilstm[aspect] = pd.DataFrame(classification_report(df_sentiment_test_true_[aspect].tolist(), df_sentiment_test_pred[aspect].tolist(), output_dict=True)).T
        sentiment_result_bilstm[aspect]['aspect'] = aspect

    output_sentiment_report = pd.DataFrame()
    for indx, val in sentiment_result_bilstm.items():
        output_sentiment_report = pd.concat([output_sentiment_report, sentiment_result_bilstm[indx]])

    return output_aspect_report

In [73]:
sentiment_result_gru = {}
# for aspect in lst_aspect:
for aspect in lst_aspect:
    sentiment_result_gru[aspect] = pd.DataFrame(classification_report(df_sentiment_test_true_[aspect].tolist(), df_sentiment_test_pred[aspect].tolist(), output_dict=True)).T
    sentiment_result_gru[aspect]['aspect'] = aspect

output_sentiment_report = pd.DataFrame()
for indx, val in sentiment_result_bilstm.items():
    output_sentiment_report = pd.concat([output_sentiment_report, sentiment_result_bilstm[indx]])

/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classif

In [74]:
output_sentiment_report

,precision,recall,f1-score,support,aspect
0,0.000000,0.000000,0.000000,3.000000,price
1,0.944206,0.890688,0.916667,247.000000,price
2,0.731481,0.868132,0.793970,91.000000,price
accuracy,0.876833,0.876833,0.876833,0.876833,price
macro avg,0.558562,0.586273,0.570212,341.000000,price
weighted avg,0.879131,0.876833,0.875859,341.000000,price
0,0.834586,0.895161,0.863813,124.000000,shipping
1,0.961818,0.963570,0.962693,549.000000,shipping
2,0.090909,0.062500,0.074074,32.000000,shipping
accuracy,0.910638,0.910638,0.910638,0.910638,shipping


In [75]:
for aspect in lst_aspect:
    print(aspect)
    print(classification_report(df_sentiment_test_true_[aspect].tolist(), df_sentiment_test_pred[aspect].tolist()))

price
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.90      0.91      0.91       247
           2       0.77      0.77      0.77        91

    accuracy                           0.87       341
   macro avg       0.56      0.56      0.56       341
weighted avg       0.86      0.87      0.86       341

shipping
              precision    recall  f1-score   support

           0       0.80      0.94      0.86       124
           1       0.96      0.97      0.96       549
           2       0.00      0.00      0.00        32

    accuracy                           0.92       705
   macro avg       0.58      0.64      0.61       705
weighted avg       0.89      0.92      0.90       705

outlook
              precision    recall  f1-score   support

           0       0.70      0.78      0.74        95
           1       0.97      0.96      0.97      1118
           2       0.30      0.28      0.29        5

/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classif

In [76]:
for i in lst_aspect:
    with open(f'{path}elmo_bigru_test_result.txt', "a") as dest:
        dest.write("Classification report for aspect: {} \n".format(i.upper()))
        dest.write(classification_report(df_sentiment_test_true_[i].tolist(), df_sentiment_test_pred[i].tolist()))

/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classif

In [77]:
for i in lst_aspect:
    with open(f'{path}elmo_bigru_val_result.txt', "a") as dest:
        dest.write("Classification report for aspect: {} \n".format(i.upper()))
        dest.write(classification_report(df_sentiment_val_true_[i].tolist(), df_sentiment_val_pred[i].tolist()))

/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classif

## 3. BiLSTM + Conv1D

In [78]:
def bilstm_aspect(vocab_aspect_size, embedding_dim, max_len, embedding_aspect_matrix, padded_aspect_train, padded_aspect_val, label_aspect_train, label_aspect_val, epochs=70):
    input = Input(shape=(max_len,))
    embed = Embedding(input_dim=vocab_aspect_size,
                      output_dim=embedding_dim,
                      embeddings_initializer=Constant(embedding_aspect_matrix),
                      input_length=max_len,
                      trainable=True)(input)
    dropout1 = SpatialDropout1D(0.2)(embed)

    lstm = Bidirectional(LSTM(units = 200, activation = 'tanh', return_sequences = True))(dropout1)
    conv = Conv1D(128, kernel_size = 2, padding = "valid", kernel_initializer = "he_uniform")(lstm)

    avg_pool1 = GlobalAveragePooling1D()(conv)
    max_pool1 = GlobalMaxPooling1D()(conv)
    
    concat = Concatenate(axis=-1)([avg_pool1, max_pool1])
    dense2 = Dense(units = 128, activation = 'relu')(concat)
    dropout1 = Dropout(rate = 0.2)(dense2)
    dense3 = Dense(units = 64, activation = 'relu')(dropout1)
    dense4 = Dense(units = 32, activation = 'relu')(dense3)
    out_aspect = Dense(units = 8, activation = 'sigmoid', name='out_aspect')(dense4)
    
    aspect_model = tf.keras.Model(inputs = input, outputs = out_aspect)
    aspect_model.compile(optimizer=Adam(learning_rate = 0.0001), loss='binary_crossentropy', metrics=['acc'])
    callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)
    
    # Fit model
    history = aspect_model.fit(x=padded_aspect_train,
                               y=label_aspect_train.to_numpy(),
                               validation_data=(padded_aspect_val, label_aspect_val.to_numpy()),
                               batch_size = 128,
                               epochs=epochs,
                               callbacks = [callback],
                               verbose=1)
    
    return aspect_model

def bilstm_polarity(aspect, vocab_sentiment_size, embedding_dim, max_len, embedding_sentiment_matrix, padded_sentiment_train, padded_sentiment_val, label_sentiment_train, label_sentiment_val, epochs=70):
    input = Input(shape=(max_len,))
    embed = Embedding(input_dim=vocab_sentiment_size[aspect],
                  output_dim=embedding_dim,
                  embeddings_initializer=Constant(embedding_sentiment_matrix[aspect]),
                  input_length=max_len,
                  trainable=True)(input)
  
    dropout1 = SpatialDropout1D(0.2)(embed)
    lstm = Bidirectional(LSTM(units = 200, activation = 'tanh', return_sequences = True))(dropout1)
    conv = Conv1D(128, kernel_size = 2, padding = "valid", kernel_initializer = "he_uniform")(lstm)

    avg_pool1 = GlobalAveragePooling1D()(conv)
    max_pool1 = GlobalMaxPooling1D()(conv)
      
      
    concat = Concatenate(axis=-1)([avg_pool1, max_pool1])

    sentiment_dense2 = Dense(128, activation='relu')(concat)
    sentiment_dropout1 = Dropout(0.2)(sentiment_dense2)
    sentiment_dense3 = Dense(64, activation='relu')(sentiment_dropout1)
    sentiment_dense4 = Dense(32, activation='relu')(sentiment_dense3)
    out_sentiment = Dense(units = 3, activation = 'softmax')(sentiment_dense4)
    
    sentiment_model_bil_con[aspect] = tf.keras.Model(inputs = input, outputs = out_sentiment)
    sentiment_model_bil_con[aspect].compile(optimizer=Adam(learning_rate = 0.0001), loss='binary_crossentropy', metrics=['acc'])
    callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)
    
    history = sentiment_model_bil_con[aspect].fit(x=padded_sentiment_train[aspect],
                                          y=label_sentiment_train[aspect],
                                          validation_data=(padded_sentiment_val[aspect], label_sentiment_val[aspect]),
                                          batch_size = 128,
                                          epochs=epochs,
                                          callbacks = [callback],
                                          verbose=1)
    
    return sentiment_model_bil_con[aspect]

In [79]:
aspect_model_bil_con = bilstm_aspect(vocab_aspect_size, embedding_dim, max_len, embedding_aspect_matrix, padded_aspect_train, padded_aspect_val, label_aspect_train, label_aspect_val)

Epoch 1/70
66/66 [==============================] - 23s 196ms/step - loss: 0.5915 - acc: 0.2793 - val_loss: 0.5016 - val_acc: 0.2842
Epoch 2/70
66/66 [==============================] - 11s 169ms/step - loss: 0.4636 - acc: 0.3852 - val_loss: 0.3907 - val_acc: 0.5235
Epoch 3/70
66/66 [==============================] - 11s 167ms/step - loss: 0.3791 - acc: 0.5089 - val_loss: 0.3186 - val_acc: 0.5513
Epoch 4/70
66/66 [==============================] - 10s 159ms/step - loss: 0.3252 - acc: 0.5313 - val_loss: 0.2854 - val_acc: 0.5940
Epoch 5/70
66/66 [==============================] - 10s 148ms/step - loss: 0.2887 - acc: 0.5459 - val_loss: 0.2549 - val_acc: 0.5459
Epoch 6/70
66/66 [==============================] - 9s 143ms/step - loss: 0.2630 - acc: 0.5399 - val_loss: 0.2405 - val_acc: 0.5769
Epoch 7/70
66/66 [==============================] - 9s 139ms/step - loss: 0.2440 - acc: 0.5493 - val_loss: 0.2289 - val_acc: 0.5780
Epoch 8/70
66/66 [==============================] - 9s 143ms/step - los

In [80]:
pred_train = aspect_model_bil_con.predict(padded_aspect_train)
pred_val = aspect_model_bil_con.predict(padded_aspect_val)
pred_test = aspect_model_bil_con.predict(padded_aspect_test)

df_train_pred = round(pd.DataFrame(pred_train), 0)
df_train_true_ = label_aspect_train
df_train_pred.columns = df_train_true_.columns

df_val_pred = round(pd.DataFrame(pred_val), 0)
df_val_true_ = label_aspect_val
df_val_pred.columns = df_val_true_.columns

df_test_pred = round(pd.DataFrame(pred_test), 0)
df_test_true_ = label_aspect_test
df_test_pred.columns = df_test_true_.columns

74/74 [==============================] - 1s 14ms/step


In [81]:
def get_pd_report(label_aspect_train, df_test_true_, df_test_pred):
    aspect_result_bilstm_report = {}
    for col in label_aspect_train.columns:
        aspect_result_bilstm = classification_report(df_test_true_[col], df_test_pred[col], output_dict=True)
        aspect_result_bilstm_report[col] = pd.DataFrame(aspect_result_bilstm).T
        aspect_result_bilstm_report[col]['aspect'] = col
        
    output_aspect_report = pd.DataFrame()
    for indx, val in aspect_result_bilstm_report.items():
        output_aspect_report = pd.concat([output_aspect_report, aspect_result_bilstm_report[indx]])
    
    return output_aspect_report 

In [82]:
aspect_result_bilstm_report = {}
for col in label_aspect_train.columns:
    print(col)
    aspect_result_bilstm = classification_report(df_test_true_[col], df_test_pred[col], output_dict=True)
    print(aspect_result_bilstm)
    
    aspect_result_bilstm_report[col] = pd.DataFrame(aspect_result_bilstm).T
    aspect_result_bilstm_report[col]['aspect'] = col
    
    output_aspect_report = pd.DataFrame()
    for indx, val in aspect_result_bilstm_report.items():
        output_aspect_report = pd.concat([output_aspect_report, aspect_result_bilstm_report[indx]])

is_price
{'0': {'precision': 0.9580896686159844, 'recall': 0.9834917458729364, 'f1-score': 0.9706245371513207, 'support': 1999}, '1': {'precision': 0.8854166666666666, 'recall': 0.7478005865102639, 'f1-score': 0.8108108108108107, 'support': 341}, 'accuracy': 0.9491452991452991, 'macro avg': {'precision': 0.9217531676413255, 'recall': 0.8656461661916002, 'f1-score': 0.8907176739810657, 'support': 2340}, 'weighted avg': {'precision': 0.9474992867079856, 'recall': 0.9491452991452991, 'f1-score': 0.947335442842725, 'support': 2340}}
is_shipping
{'0': {'precision': 0.9821208384710234, 'recall': 0.9743119266055046, 'f1-score': 0.9782007982806263, 'support': 1635}, '1': {'precision': 0.9415041782729805, 'recall': 0.9588652482269504, 'f1-score': 0.9501054111033029, 'support': 705}, 'accuracy': 0.9696581196581197, 'macro avg': {'precision': 0.9618125083720019, 'recall': 0.9665885874162274, 'f1-score': 0.9641531046919647, 'support': 2340}, 'weighted avg': {'precision': 0.9698837677703309, 'recal

In [83]:
for col in label_aspect_train.columns:
    print(col)
    aspect_result = classification_report(df_test_true_[col], df_test_pred[col])

    print(aspect_result)

is_price
              precision    recall  f1-score   support

           0       0.96      0.98      0.97      1999
           1       0.89      0.75      0.81       341

    accuracy                           0.95      2340
   macro avg       0.92      0.87      0.89      2340
weighted avg       0.95      0.95      0.95      2340

is_shipping
              precision    recall  f1-score   support

           0       0.98      0.97      0.98      1635
           1       0.94      0.96      0.95       705

    accuracy                           0.97      2340
   macro avg       0.96      0.97      0.96      2340
weighted avg       0.97      0.97      0.97      2340

is_outlook
              precision    recall  f1-score   support

           0       0.95      0.87      0.91      1069
           1       0.90      0.96      0.93      1271

    accuracy                           0.92      2340
   macro avg       0.92      0.92      0.92      2340
weighted avg       0.92      0.92      0.9

In [84]:
path = '/kaggle/working/'

In [85]:
save_result_pred_aspect(df_test_true_, df_test_pred, data_eval = 'test', model_type = 'bilstm_con')
save_result_pred_aspect(df_val_true_, df_val_pred, data_eval = 'eval', model_type = 'bilstm_con')

In [86]:
lst_aspect = []
for col in label_aspect_train.columns.values:
    if col!='is_others':
        lst_aspect.append(col.split('_')[1])

In [87]:
sentiment_model_bil_con = {}
for aspect in lst_aspect:
    sentiment_model_bil_con[aspect] = bilstm_polarity(aspect, vocab_sentiment_size, embedding_dim, max_len, embedding_sentiment_matrix, padded_sentiment_train, padded_sentiment_val, label_sentiment_train, label_sentiment_val)

Epoch 1/70
10/10 [==============================] - 8s 301ms/step - loss: 0.6680 - acc: 0.5372 - val_loss: 0.5658 - val_acc: 0.7681
Epoch 2/70
10/10 [==============================] - 2s 191ms/step - loss: 0.5349 - acc: 0.7426 - val_loss: 0.4600 - val_acc: 0.7681
Epoch 3/70
10/10 [==============================] - 2s 193ms/step - loss: 0.4634 - acc: 0.7426 - val_loss: 0.4222 - val_acc: 0.7681
Epoch 4/70
10/10 [==============================] - 2s 183ms/step - loss: 0.4338 - acc: 0.7426 - val_loss: 0.4015 - val_acc: 0.7681
Epoch 5/70
10/10 [==============================] - 2s 182ms/step - loss: 0.4094 - acc: 0.7426 - val_loss: 0.3706 - val_acc: 0.7681
Epoch 6/70
10/10 [==============================] - 2s 185ms/step - loss: 0.3695 - acc: 0.7426 - val_loss: 0.3352 - val_acc: 0.7681
Epoch 7/70
10/10 [==============================] - 2s 186ms/step - loss: 0.3310 - acc: 0.7426 - val_loss: 0.2962 - val_acc: 0.7681
Epoch 8/70
10/10 [==============================] - 2s 159ms/step - loss: 0.

In [88]:
df_sentiment_train_pred = {}
df_sentiment_train_true_ = {}

df_sentiment_val_true_ = {}
df_sentiment_val_pred = {}

df_sentiment_test_true_ = {}
df_sentiment_test_pred = {}


for aspect in lst_aspect:
    pred_sentiment_train = sentiment_model_bil_con[aspect].predict(padded_sentiment_train[aspect])
    pred_sentiment_val = sentiment_model_bil_con[aspect].predict(padded_sentiment_val[aspect])
    pred_sentiment_test = sentiment_model_bil_con[aspect].predict(padded_sentiment_test[aspect])
    
    df_sentiment_train_pred[aspect] = np.argmax(pred_sentiment_train, axis=1)
    df_sentiment_train_true_[aspect] = np.argmax(label_sentiment_train[aspect], axis = 1)
    
    df_sentiment_val_pred[aspect] = np.argmax(pred_sentiment_val, axis=1)
    df_sentiment_val_true_[aspect] = np.argmax(label_sentiment_val[aspect], axis = 1)
    
    df_sentiment_test_pred[aspect] = np.argmax(pred_sentiment_test, axis=1)
    df_sentiment_test_true_[aspect] = np.argmax(label_sentiment_test[aspect], axis=1)

15/15 [==============================] - 0s 13ms/step


In [89]:
def get_pd_report_sentiment(df_sentiment_test_true_, df_sentiment_test_pred):
    sentiment_result_bilstm = {}
    for aspect in lst_aspect:
        sentiment_result_bilstm[aspect] = pd.DataFrame(classification_report(df_sentiment_test_true_[aspect].tolist(), df_sentiment_test_pred[aspect].tolist(), output_dict=True)).T
        sentiment_result_bilstm[aspect]['aspect'] = aspect

    output_sentiment_report = pd.DataFrame()
    for indx, val in sentiment_result_bilstm.items():
        output_sentiment_report = pd.concat([output_sentiment_report, sentiment_result_bilstm[indx]])

    return output_aspect_report

In [90]:
sentiment_result_bilstm = {}
# for aspect in lst_aspect:
for aspect in lst_aspect:
    sentiment_result_bilstm[aspect] = pd.DataFrame(classification_report(df_sentiment_test_true_[aspect].tolist(), df_sentiment_test_pred[aspect].tolist(), output_dict=True)).T
    sentiment_result_bilstm[aspect]['aspect'] = aspect

output_sentiment_report = pd.DataFrame()
for indx, val in sentiment_result_bilstm.items():
    output_sentiment_report = pd.concat([output_sentiment_report, sentiment_result_bilstm[indx]])

/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classif

In [91]:
output_sentiment_report

,precision,recall,f1-score,support,aspect
0,0.000000,0.000000,0.000000,3.000000,price
1,0.920168,0.886640,0.903093,247.000000,price
2,0.708738,0.802198,0.752577,91.000000,price
accuracy,0.856305,0.856305,0.856305,0.856305,price
macro avg,0.542969,0.562946,0.551890,341.000000,price
weighted avg,0.855650,0.856305,0.854981,341.000000,price
0,0.808511,0.919355,0.860377,124.000000,shipping
1,0.958855,0.976321,0.967509,549.000000,shipping
2,0.200000,0.031250,0.054054,32.000000,shipping
accuracy,0.923404,0.923404,0.923404,0.923404,shipping


In [92]:
for aspect in lst_aspect:
    print(aspect)
    print(classification_report(df_sentiment_test_true_[aspect].tolist(), df_sentiment_test_pred[aspect].tolist()))

price
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.92      0.89      0.90       247
           2       0.71      0.80      0.75        91

    accuracy                           0.86       341
   macro avg       0.54      0.56      0.55       341
weighted avg       0.86      0.86      0.85       341

shipping
              precision    recall  f1-score   support

           0       0.81      0.92      0.86       124
           1       0.96      0.98      0.97       549
           2       0.20      0.03      0.05        32

    accuracy                           0.92       705
   macro avg       0.66      0.64      0.63       705
weighted avg       0.90      0.92      0.91       705

outlook
              precision    recall  f1-score   support

           0       0.74      0.77      0.75        95
           1       0.97      0.97      0.97      1118
           2       0.38      0.33      0.35        5

/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classif

In [93]:
for i in lst_aspect:
    with open(f'{path}elmo_bilstm_con_test_result.txt', "a") as dest:
        dest.write("Classification report for aspect: {} \n".format(i.upper()))
        dest.write(classification_report(df_sentiment_test_true_[i].tolist(), df_sentiment_test_pred[i].tolist()))

/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classif

In [94]:
for i in lst_aspect:
    with open(f'{path}elmo_bilstm_con_val_result.txt', "a") as dest:
        dest.write("Classification report for aspect: {} \n".format(i.upper()))
        dest.write(classification_report(df_sentiment_val_true_[i].tolist(), df_sentiment_val_pred[i].tolist()))

/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classif

## 4. BiGRU + Conv1D

In [95]:
def gru_aspect(vocab_aspect_size, embedding_dim, max_len, embedding_aspect_matrix, padded_aspect_train, padded_aspect_val, label_aspect_train, label_aspect_val, epochs=70):
    input = Input(shape=(max_len,))
    embed = Embedding(input_dim=vocab_aspect_size,
                      output_dim=embedding_dim,
                      embeddings_initializer=Constant(embedding_aspect_matrix),
                      input_length=max_len,
                      trainable=True)(input)
    dropout1 = SpatialDropout1D(0.2)(embed)

    lstm = Bidirectional(GRU(units = 200, activation = 'tanh', return_sequences = True))(dropout1)
    conv = Conv1D(128, kernel_size = 2, padding = "valid", kernel_initializer = "he_uniform")(lstm)

    avg_pool1 = GlobalAveragePooling1D()(conv)
    max_pool1 = GlobalMaxPooling1D()(conv)
    
    concat = Concatenate(axis=-1)([avg_pool1, max_pool1])
    dense2 = Dense(units = 128, activation = 'relu')(concat)
    dropout1 = Dropout(rate = 0.2)(dense2)
    dense3 = Dense(units = 64, activation = 'relu')(dropout1)
    dense4 = Dense(units = 32, activation = 'relu')(dense3)
    out_aspect = Dense(units = 8, activation = 'sigmoid', name='out_aspect')(dense4)
    
    aspect_model = tf.keras.Model(inputs = input, outputs = out_aspect)
    aspect_model.compile(optimizer=Adam(learning_rate = 0.0001), loss='binary_crossentropy', metrics=['acc'])
    callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)
    
    # Fit model
    history = aspect_model.fit(x=padded_aspect_train,
                               y=label_aspect_train.to_numpy(),
                               validation_data=(padded_aspect_val, label_aspect_val.to_numpy()),
                               batch_size = 128,
                               epochs=epochs,
                               callbacks = [callback],
                               verbose=1)
    
    return aspect_model

def gru_polarity(aspect, vocab_sentiment_size, embedding_dim, max_len, embedding_sentiment_matrix, padded_sentiment_train, padded_sentiment_val, label_sentiment_train, label_sentiment_val, epochs=70):
    input = Input(shape=(max_len,))
    embed = Embedding(input_dim=vocab_sentiment_size[aspect],
                  output_dim=embedding_dim,
                  embeddings_initializer=Constant(embedding_sentiment_matrix[aspect]),
                  input_length=max_len,
                  trainable=True)(input)
  
    dropout1 = SpatialDropout1D(0.2)(embed)
    lstm = Bidirectional(GRU(units = 200, activation = 'tanh', return_sequences = True))(dropout1)
    conv = Conv1D(128, kernel_size = 2, padding = "valid", kernel_initializer = "he_uniform")(lstm)

    avg_pool1 = GlobalAveragePooling1D()(conv)
    max_pool1 = GlobalMaxPooling1D()(conv)
      
      
    concat = Concatenate(axis=-1)([avg_pool1, max_pool1])

    sentiment_dense2 = Dense(128, activation='relu')(concat)
    sentiment_dropout1 = Dropout(0.2)(sentiment_dense2)
    sentiment_dense3 = Dense(64, activation='relu')(sentiment_dropout1)
    sentiment_dense4 = Dense(32, activation='relu')(sentiment_dense3)
    out_sentiment = Dense(units = 3, activation = 'softmax')(sentiment_dense4)
    
    sentiment_model_big_con[aspect] = tf.keras.Model(inputs = input, outputs = out_sentiment)
    sentiment_model_big_con[aspect].compile(optimizer=Adam(learning_rate = 0.0001), loss='binary_crossentropy', metrics=['acc'])
    callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)
    
    history = sentiment_model_big_con[aspect].fit(x=padded_sentiment_train[aspect],
                                          y=label_sentiment_train[aspect],
                                          validation_data=(padded_sentiment_val[aspect], label_sentiment_val[aspect]),
                                          batch_size = 128,
                                          epochs=epochs,
                                          callbacks = [callback],
                                          verbose=1)
    
    return sentiment_model_big_con[aspect]

In [105]:
aspect_model_big_con = gru_aspect(vocab_aspect_size, embedding_dim, max_len, embedding_aspect_matrix, padded_aspect_train, padded_aspect_val, label_aspect_train, label_aspect_val)

Epoch 1/70
66/66 [==============================] - 18s 190ms/step - loss: 0.5504 - acc: 0.2807 - val_loss: 0.4711 - val_acc: 0.3066
Epoch 2/70
66/66 [==============================] - 11s 164ms/step - loss: 0.4488 - acc: 0.3934 - val_loss: 0.3983 - val_acc: 0.5085
Epoch 3/70
66/66 [==============================] - 9s 141ms/step - loss: 0.3854 - acc: 0.4789 - val_loss: 0.3401 - val_acc: 0.5449
Epoch 4/70
66/66 [==============================] - 9s 139ms/step - loss: 0.3293 - acc: 0.5058 - val_loss: 0.2927 - val_acc: 0.5620
Epoch 5/70
66/66 [==============================] - 9s 132ms/step - loss: 0.2901 - acc: 0.5316 - val_loss: 0.2606 - val_acc: 0.5652
Epoch 6/70
66/66 [==============================] - 8s 120ms/step - loss: 0.2610 - acc: 0.5566 - val_loss: 0.2410 - val_acc: 0.5940
Epoch 7/70
66/66 [==============================] - 8s 118ms/step - loss: 0.2389 - acc: 0.5689 - val_loss: 0.2275 - val_acc: 0.5983
Epoch 8/70
66/66 [==============================] - 8s 116ms/step - loss: 

In [106]:
pred_train = aspect_model_big_con.predict(padded_aspect_train)
pred_val = aspect_model_big_con.predict(padded_aspect_val)
pred_test = aspect_model_big_con.predict(padded_aspect_test)

df_train_pred = round(pd.DataFrame(pred_train), 0)
df_train_true_ = label_aspect_train
df_train_pred.columns = df_train_true_.columns

df_val_pred = round(pd.DataFrame(pred_val), 0)
df_val_true_ = label_aspect_val
df_val_pred.columns = df_val_true_.columns

df_test_pred = round(pd.DataFrame(pred_test), 0)
df_test_true_ = label_aspect_test
df_test_pred.columns = df_test_true_.columns

74/74 [==============================] - 1s 12ms/step


In [107]:
def get_pd_report(label_aspect_train, df_test_true_, df_test_pred):
    aspect_result_gru_report = {}
    for col in label_aspect_train.columns:
        aspect_result_gru_report = classification_report(df_test_true_[col], df_test_pred[col], output_dict=True)
        aspect_result_gru_report[col] = pd.DataFrame(aspect_result_bilstm).T
        aspect_result_gru_report[col]['aspect'] = col
        
    output_aspect_report = pd.DataFrame()
    for indx, val in aspect_result_gru_report.items():
        output_aspect_report = pd.concat([output_aspect_report, aspect_result_gru_report[indx]])
    
    return output_aspect_report 

In [108]:
aspect_result_gru_report = {}
for col in label_aspect_train.columns:
    print(col)
    aspect_result_gru = classification_report(df_test_true_[col], df_test_pred[col], output_dict=True)
    print(aspect_result_gru)
    
    aspect_result_gru_report[col] = pd.DataFrame(aspect_result_gru).T
    aspect_result_gru_report[col]['aspect'] = col
    
    output_aspect_report = pd.DataFrame()
    for indx, val in aspect_result_gru_report.items():
        output_aspect_report = pd.concat([output_aspect_report, aspect_result_gru_report[indx]])

is_price
{'0': {'precision': 0.9656188605108055, 'recall': 0.9834917458729364, 'f1-score': 0.9744733581164807, 'support': 1999}, '1': {'precision': 0.8914473684210527, 'recall': 0.7947214076246334, 'f1-score': 0.8403100775193798, 'support': 341}, 'accuracy': 0.955982905982906, 'macro avg': {'precision': 0.928533114465929, 'recall': 0.8891065767487849, 'f1-score': 0.9073917178179303, 'support': 2340}, 'weighted avg': {'precision': 0.9548101088857601, 'recall': 0.955982905982906, 'f1-score': 0.9549222133798948, 'support': 2340}}
is_shipping
{'0': {'precision': 0.9808286951144094, 'recall': 0.9700305810397554, 'f1-score': 0.97539975399754, 'support': 1635}, '1': {'precision': 0.9322268326417704, 'recall': 0.9560283687943263, 'f1-score': 0.9439775910364147, 'support': 705}, 'accuracy': 0.9658119658119658, 'macro avg': {'precision': 0.9565277638780899, 'recall': 0.9630294749170408, 'f1-score': 0.9596886725169773, 'support': 2340}, 'weighted avg': {'precision': 0.9661858262925245, 'recall': 

In [109]:
for col in label_aspect_train.columns:
    print(col)
    aspect_result = classification_report(df_test_true_[col], df_test_pred[col])

    print(aspect_result)

is_price
              precision    recall  f1-score   support

           0       0.97      0.98      0.97      1999
           1       0.89      0.79      0.84       341

    accuracy                           0.96      2340
   macro avg       0.93      0.89      0.91      2340
weighted avg       0.95      0.96      0.95      2340

is_shipping
              precision    recall  f1-score   support

           0       0.98      0.97      0.98      1635
           1       0.93      0.96      0.94       705

    accuracy                           0.97      2340
   macro avg       0.96      0.96      0.96      2340
weighted avg       0.97      0.97      0.97      2340

is_outlook
              precision    recall  f1-score   support

           0       0.90      0.94      0.92      1069
           1       0.94      0.92      0.93      1271

    accuracy                           0.92      2340
   macro avg       0.92      0.93      0.92      2340
weighted avg       0.93      0.92      0.9

In [110]:
path = '/kaggle/working/'

In [111]:
save_result_pred_aspect(df_test_true_, df_test_pred, data_eval = 'test', model_type = 'bigru_con')
save_result_pred_aspect(df_val_true_, df_val_pred, data_eval = 'eval', model_type = 'bigru_con')

In [112]:
lst_aspect = []
for col in label_aspect_train.columns.values:
    if col!='is_others':
        lst_aspect.append(col.split('_')[1])

In [113]:
sentiment_model_big_con = {}
for aspect in lst_aspect:
    sentiment_model_big_con[aspect] = gru_polarity(aspect, vocab_sentiment_size, embedding_dim, max_len, embedding_sentiment_matrix, padded_sentiment_train, padded_sentiment_val, label_sentiment_train, label_sentiment_val)

Epoch 1/70
10/10 [==============================] - 8s 267ms/step - loss: 0.5278 - acc: 0.7330 - val_loss: 0.4123 - val_acc: 0.7681
Epoch 2/70
10/10 [==============================] - 2s 181ms/step - loss: 0.4319 - acc: 0.7426 - val_loss: 0.3728 - val_acc: 0.7681
Epoch 3/70
10/10 [==============================] - 2s 158ms/step - loss: 0.3904 - acc: 0.7426 - val_loss: 0.3330 - val_acc: 0.7681
Epoch 4/70
10/10 [==============================] - 2s 184ms/step - loss: 0.3477 - acc: 0.7474 - val_loss: 0.2933 - val_acc: 0.7754
Epoch 5/70
10/10 [==============================] - 2s 170ms/step - loss: 0.3018 - acc: 0.7786 - val_loss: 0.2513 - val_acc: 0.8261
Epoch 6/70
10/10 [==============================] - 2s 169ms/step - loss: 0.2568 - acc: 0.8385 - val_loss: 0.2214 - val_acc: 0.8913
Epoch 7/70
10/10 [==============================] - 2s 159ms/step - loss: 0.2215 - acc: 0.8641 - val_loss: 0.2080 - val_acc: 0.8841
Epoch 8/70
10/10 [==============================] - 1s 110ms/step - loss: 0.

In [114]:
df_sentiment_train_pred = {}
df_sentiment_train_true_ = {}

df_sentiment_val_true_ = {}
df_sentiment_val_pred = {}

df_sentiment_test_true_ = {}
df_sentiment_test_pred = {}


for aspect in lst_aspect:
    pred_sentiment_train = sentiment_model_big_con[aspect].predict(padded_sentiment_train[aspect])
    pred_sentiment_val = sentiment_model_big_con[aspect].predict(padded_sentiment_val[aspect])
    pred_sentiment_test = sentiment_model_big_con[aspect].predict(padded_sentiment_test[aspect])
    
    df_sentiment_train_pred[aspect] = np.argmax(pred_sentiment_train, axis=1)
    df_sentiment_train_true_[aspect] = np.argmax(label_sentiment_train[aspect], axis = 1)
    
    df_sentiment_val_pred[aspect] = np.argmax(pred_sentiment_val, axis=1)
    df_sentiment_val_true_[aspect] = np.argmax(label_sentiment_val[aspect], axis = 1)
    
    df_sentiment_test_pred[aspect] = np.argmax(pred_sentiment_test, axis=1)
    df_sentiment_test_true_[aspect] = np.argmax(label_sentiment_test[aspect], axis=1)

15/15 [==============================] - 0s 12ms/step


In [115]:
def get_pd_report_sentiment(df_sentiment_test_true_, df_sentiment_test_pred):
    sentiment_result_bilstm = {}
    for aspect in lst_aspect:
        sentiment_result_bilstm[aspect] = pd.DataFrame(classification_report(df_sentiment_test_true_[aspect].tolist(), df_sentiment_test_pred[aspect].tolist(), output_dict=True)).T
        sentiment_result_bilstm[aspect]['aspect'] = aspect

    output_sentiment_report = pd.DataFrame()
    for indx, val in sentiment_result_bilstm.items():
        output_sentiment_report = pd.concat([output_sentiment_report, sentiment_result_bilstm[indx]])

    return output_aspect_report

In [116]:
sentiment_result_gru = {}
# for aspect in lst_aspect:
for aspect in lst_aspect:
    sentiment_result_gru[aspect] = pd.DataFrame(classification_report(df_sentiment_test_true_[aspect].tolist(), df_sentiment_test_pred[aspect].tolist(), output_dict=True)).T
    sentiment_result_gru[aspect]['aspect'] = aspect

output_sentiment_report = pd.DataFrame()
for indx, val in sentiment_result_bilstm.items():
    output_sentiment_report = pd.concat([output_sentiment_report, sentiment_result_bilstm[indx]])

/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classif

In [117]:
output_sentiment_report

,precision,recall,f1-score,support,aspect
0,0.000000,0.000000,0.000000,3.000000,price
1,0.920168,0.886640,0.903093,247.000000,price
2,0.708738,0.802198,0.752577,91.000000,price
accuracy,0.856305,0.856305,0.856305,0.856305,price
macro avg,0.542969,0.562946,0.551890,341.000000,price
weighted avg,0.855650,0.856305,0.854981,341.000000,price
0,0.808511,0.919355,0.860377,124.000000,shipping
1,0.958855,0.976321,0.967509,549.000000,shipping
2,0.200000,0.031250,0.054054,32.000000,shipping
accuracy,0.923404,0.923404,0.923404,0.923404,shipping


In [118]:
for aspect in lst_aspect:
    print(aspect)
    print(classification_report(df_sentiment_test_true_[aspect].tolist(), df_sentiment_test_pred[aspect].tolist()))

price
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.93      0.91      0.92       247
           2       0.76      0.82      0.79        91

    accuracy                           0.88       341
   macro avg       0.56      0.58      0.57       341
weighted avg       0.87      0.88      0.87       341

shipping
              precision    recall  f1-score   support

           0       0.83      0.95      0.89       124
           1       0.96      0.98      0.97       549
           2       0.00      0.00      0.00        32

    accuracy                           0.93       705
   macro avg       0.60      0.64      0.62       705
weighted avg       0.90      0.93      0.91       705

outlook
              precision    recall  f1-score   support

           0       0.75      0.80      0.77        95
           1       0.97      0.97      0.97      1118
           2       0.34      0.33      0.33        5

/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classif

In [119]:
for i in lst_aspect:
    with open(f'{path}elmo_bigru_con_test_result.txt', "a") as dest:
        dest.write("Classification report for aspect: {} \n".format(i.upper()))
        dest.write(classification_report(df_sentiment_test_true_[i].tolist(), df_sentiment_test_pred[i].tolist()))

/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classif

In [120]:
for i in lst_aspect:
    with open(f'{path}elmo_bigru_con_val_result.txt', "a") as dest:
        dest.write("Classification report for aspect: {} \n".format(i.upper()))
        dest.write(classification_report(df_sentiment_val_true_[i].tolist(), df_sentiment_val_pred[i].tolist()))

/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classif

## 5. BiLSTM + BiGRU + Conv1D

In [121]:
def bilstm_aspect(vocab_aspect_size, embedding_dim, max_len, embedding_aspect_matrix, padded_aspect_train, padded_aspect_val, label_aspect_train, label_aspect_val, epochs=70):
    input = Input(shape=(max_len,))
    embed = Embedding(input_dim=vocab_aspect_size,
                  output_dim=embedding_dim,
                  embeddings_initializer=Constant(embedding_aspect_matrix),
                  input_length=max_len,
                  trainable=True)(input)

    dropout1 = SpatialDropout1D(0.2)(embed)

    lstm = Bidirectional(LSTM(units = 200, activation = 'tanh', return_sequences = True))(dropout1)
    conv_lstm = Conv1D(128, kernel_size = 2, padding = "valid", kernel_initializer = "he_uniform")(lstm)

    gru = Bidirectional(GRU(units = 200, activation = 'tanh', return_sequences = True))(dropout1)
    conv_gru = Conv1D(128, kernel_size = 2, padding = "valid", kernel_initializer = "he_uniform")(gru)


    avg_pool1 = GlobalAveragePooling1D()(conv_lstm)
    max_pool1 = GlobalMaxPooling1D()(conv_lstm)

    avg_pool2 = GlobalAveragePooling1D()(conv_gru)
    max_pool2 = GlobalMaxPooling1D()(conv_gru)
      
      
    concat = Concatenate(axis=-1)([avg_pool1, max_pool1, avg_pool2, max_pool2])
 
    aspect_dense1 = Dense(units = 256, activation = 'relu')(concat)
    aspect_dense2 = Dense(128, activation='relu')(aspect_dense1)
    aspect_dropout1 = Dropout(0.2)(aspect_dense2)
    aspect_dense3 = Dense(64, activation='relu')(aspect_dropout1)
    aspect_dense4 = Dense(32, activation='relu')(aspect_dense3)
    aspect_dense5 = Dense(8, activation='sigmoid')(aspect_dense4)

    
    aspect_model = tf.keras.Model(inputs = input, outputs = aspect_dense5)
    aspect_model.compile(optimizer=Adam(learning_rate = 0.0001), loss='binary_crossentropy', metrics=['acc'])
    callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)
    
    # Fit model
    history = aspect_model.fit(x=padded_aspect_train,
                               y=label_aspect_train.to_numpy(),
                               validation_data=(padded_aspect_val, label_aspect_val.to_numpy()),
                               batch_size = 128,
                               epochs=epochs,
                               callbacks = [callback],
                               verbose=1)
    
    return aspect_model

def bilstm_polarity(aspect, vocab_sentiment_size, embedding_dim, max_len, embedding_sentiment_matrix, padded_sentiment_train, padded_sentiment_val, label_sentiment_train, label_sentiment_val, epochs=70):
    input = Input(shape=(max_len,))
    embed = Embedding(input_dim=vocab_sentiment_size[aspect],
                  output_dim=embedding_dim,
                  embeddings_initializer=Constant(embedding_sentiment_matrix[aspect]),
                  input_length=max_len,
                  trainable=True)(input)
  
    dropout1 = SpatialDropout1D(0.2)(embed)

    lstm = Bidirectional(LSTM(units = 200, activation = 'tanh', return_sequences = True))(dropout1)
    conv_lstm = Conv1D(128, kernel_size = 2, padding = "valid", kernel_initializer = "he_uniform")(lstm)

    gru = Bidirectional(GRU(units = 200, activation = 'tanh', return_sequences = True))(dropout1)
    conv_gru = Conv1D(128, kernel_size = 2, padding = "valid", kernel_initializer = "he_uniform")(gru)


    avg_pool1 = GlobalAveragePooling1D()(conv_lstm)
    max_pool1 = GlobalMaxPooling1D()(conv_lstm)

    avg_pool2 = GlobalAveragePooling1D()(conv_gru)
    max_pool2 = GlobalMaxPooling1D()(conv_gru)
      
      
    concat = Concatenate(axis=-1)([avg_pool1, max_pool1, avg_pool2, max_pool2])

    sentiment_dense1 = Dense(units = 256, activation = 'relu')(concat)
    sentiment_dense2 = Dense(128, activation='relu')(sentiment_dense1)
    sentiment_dropout1 = Dropout(0.2)(sentiment_dense2)
    sentiment_dense3 = Dense(64, activation='relu')(sentiment_dropout1)
    sentiment_dense4 = Dense(32, activation='relu')(sentiment_dense3)
    out_sentiment = Dense(units = 3, activation = 'softmax')(sentiment_dense4)
    
    sentiment_model_lgc[aspect] = tf.keras.Model(inputs = input, outputs = out_sentiment)
    sentiment_model_lgc[aspect].compile(optimizer=Adam(learning_rate = 0.0001), loss='binary_crossentropy', metrics=['acc'])
    callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)
    
    history = sentiment_model_lgc[aspect].fit(x=padded_sentiment_train[aspect],
                                          y=label_sentiment_train[aspect],
                                          validation_data=(padded_sentiment_val[aspect], label_sentiment_val[aspect]),
                                          batch_size = 128,
                                          epochs=epochs,
                                          callbacks = [callback],
                                          verbose=1)
    
    return sentiment_model_lgc[aspect]

In [122]:
aspect_model_lgc = bilstm_aspect(vocab_aspect_size, embedding_dim, max_len, embedding_aspect_matrix, padded_aspect_train, padded_aspect_val, label_aspect_train, label_aspect_val)

Epoch 1/70
66/66 [==============================] - 29s 304ms/step - loss: 0.5298 - acc: 0.3075 - val_loss: 0.4518 - val_acc: 0.4209
Epoch 2/70
66/66 [==============================] - 17s 263ms/step - loss: 0.4288 - acc: 0.4721 - val_loss: 0.3869 - val_acc: 0.5085
Epoch 3/70
66/66 [==============================] - 17s 261ms/step - loss: 0.3741 - acc: 0.5147 - val_loss: 0.3290 - val_acc: 0.5011
Epoch 4/70
66/66 [==============================] - 16s 248ms/step - loss: 0.3191 - acc: 0.5387 - val_loss: 0.2770 - val_acc: 0.5994
Epoch 5/70
66/66 [==============================] - 16s 239ms/step - loss: 0.2690 - acc: 0.5780 - val_loss: 0.2448 - val_acc: 0.5865
Epoch 6/70
66/66 [==============================] - 16s 239ms/step - loss: 0.2412 - acc: 0.5780 - val_loss: 0.2225 - val_acc: 0.6015
Epoch 7/70
66/66 [==============================] - 16s 237ms/step - loss: 0.2203 - acc: 0.5943 - val_loss: 0.2140 - val_acc: 0.5908
Epoch 8/70
66/66 [==============================] - 16s 236ms/step - 

In [123]:
pred_train = aspect_model_lgc.predict(padded_aspect_train)
pred_val = aspect_model_lgc.predict(padded_aspect_val)
pred_test = aspect_model_lgc.predict(padded_aspect_test)

df_train_pred = round(pd.DataFrame(pred_train), 0)
df_train_true_ = label_aspect_train
df_train_pred.columns = df_train_true_.columns

df_val_pred = round(pd.DataFrame(pred_val), 0)
df_val_true_ = label_aspect_val
df_val_pred.columns = df_val_true_.columns

df_test_pred = round(pd.DataFrame(pred_test), 0)
df_test_true_ = label_aspect_test
df_test_pred.columns = df_test_true_.columns

74/74 [==============================] - 2s 25ms/step


In [124]:
def get_pd_report(label_aspect_train, df_test_true_, df_test_pred):
    aspect_result_bilstm_report = {}
    for col in label_aspect_train.columns:
        aspect_result_bilstm = classification_report(df_test_true_[col], df_test_pred[col], output_dict=True)
        aspect_result_bilstm_report[col] = pd.DataFrame(aspect_result_bilstm).T
        aspect_result_bilstm_report[col]['aspect'] = col
        
    output_aspect_report = pd.DataFrame()
    for indx, val in aspect_result_bilstm_report.items():
        output_aspect_report = pd.concat([output_aspect_report, aspect_result_bilstm_report[indx]])
    
    return output_aspect_report 

In [125]:
aspect_result_bilstm_report = {}
for col in label_aspect_train.columns:
    print(col)
    aspect_result_bilstm = classification_report(df_test_true_[col], df_test_pred[col], output_dict=True)
    print(aspect_result_bilstm)
    
    aspect_result_bilstm_report[col] = pd.DataFrame(aspect_result_bilstm).T
    aspect_result_bilstm_report[col]['aspect'] = col
    
    output_aspect_report = pd.DataFrame()
    for indx, val in aspect_result_bilstm_report.items():
        output_aspect_report = pd.concat([output_aspect_report, aspect_result_bilstm_report[indx]])

is_price
{'0': {'precision': 0.9572400388726919, 'recall': 0.9854927463731866, 'f1-score': 0.9711609563717033, 'support': 1999}, '1': {'precision': 0.8971631205673759, 'recall': 0.7419354838709677, 'f1-score': 0.8121990369181381, 'support': 341}, 'accuracy': 0.95, 'macro avg': {'precision': 0.9272015797200339, 'recall': 0.8637141151220772, 'f1-score': 0.8916799966449207, 'support': 2340}, 'weighted avg': {'precision': 0.9484852400940113, 'recall': 0.95, 'f1-score': 0.9479959928957777, 'support': 2340}}
is_shipping
{'0': {'precision': 0.9809113300492611, 'recall': 0.9743119266055046, 'f1-score': 0.9776004909481436, 'support': 1635}, '1': {'precision': 0.9413407821229051, 'recall': 0.9560283687943263, 'f1-score': 0.9486277269528501, 'support': 705}, 'accuracy': 0.9688034188034188, 'macro avg': {'precision': 0.961126056086083, 'recall': 0.9651701476999155, 'f1-score': 0.9631141089504969, 'support': 2340}, 'weighted avg': {'precision': 0.9689894341996538, 'recall': 0.9688034188034188, 'f1-

In [126]:
for col in label_aspect_train.columns:
    print(col)
    aspect_result = classification_report(df_test_true_[col], df_test_pred[col])

    print(aspect_result)

is_price
              precision    recall  f1-score   support

           0       0.96      0.99      0.97      1999
           1       0.90      0.74      0.81       341

    accuracy                           0.95      2340
   macro avg       0.93      0.86      0.89      2340
weighted avg       0.95      0.95      0.95      2340

is_shipping
              precision    recall  f1-score   support

           0       0.98      0.97      0.98      1635
           1       0.94      0.96      0.95       705

    accuracy                           0.97      2340
   macro avg       0.96      0.97      0.96      2340
weighted avg       0.97      0.97      0.97      2340

is_outlook
              precision    recall  f1-score   support

           0       0.90      0.92      0.91      1069
           1       0.93      0.92      0.92      1271

    accuracy                           0.92      2340
   macro avg       0.92      0.92      0.92      2340
weighted avg       0.92      0.92      0.9

In [127]:
path = '/kaggle/working/'

In [128]:
save_result_pred_aspect(df_test_true_, df_test_pred, data_eval = 'test', model_type = 'bilstm_bigru_con')
save_result_pred_aspect(df_val_true_, df_val_pred, data_eval = 'eval', model_type = 'bilstm_bigru_con')

In [129]:
lst_aspect = []
for col in label_aspect_train.columns.values:
    if col!='is_others':
        lst_aspect.append(col.split('_')[1])

In [130]:
sentiment_model_lgc = {}
for aspect in lst_aspect:
    sentiment_model_lgc[aspect] = bilstm_polarity(aspect, vocab_sentiment_size, embedding_dim, max_len, embedding_sentiment_matrix, padded_sentiment_train, padded_sentiment_val, label_sentiment_train, label_sentiment_val)

Epoch 1/70
10/10 [==============================] - 13s 449ms/step - loss: 0.6779 - acc: 0.2470 - val_loss: 0.6236 - val_acc: 0.2246
Epoch 2/70
10/10 [==============================] - 3s 292ms/step - loss: 0.6056 - acc: 0.3221 - val_loss: 0.5464 - val_acc: 0.7826
Epoch 3/70
10/10 [==============================] - 3s 288ms/step - loss: 0.5382 - acc: 0.5548 - val_loss: 0.4657 - val_acc: 0.7681
Epoch 4/70
10/10 [==============================] - 3s 270ms/step - loss: 0.4591 - acc: 0.7258 - val_loss: 0.3889 - val_acc: 0.7681
Epoch 5/70
10/10 [==============================] - 3s 257ms/step - loss: 0.4047 - acc: 0.7378 - val_loss: 0.3565 - val_acc: 0.7681
Epoch 6/70
10/10 [==============================] - 3s 262ms/step - loss: 0.3704 - acc: 0.7426 - val_loss: 0.3261 - val_acc: 0.7681
Epoch 7/70
10/10 [==============================] - 3s 251ms/step - loss: 0.3342 - acc: 0.7538 - val_loss: 0.2775 - val_acc: 0.7826
Epoch 8/70
10/10 [==============================] - 3s 295ms/step - loss: 0

In [131]:
df_sentiment_train_pred = {}
df_sentiment_train_true_ = {}

df_sentiment_val_true_ = {}
df_sentiment_val_pred = {}

df_sentiment_test_true_ = {}
df_sentiment_test_pred = {}


for aspect in lst_aspect:
    pred_sentiment_train = sentiment_model_lgc[aspect].predict(padded_sentiment_train[aspect])
    pred_sentiment_val = sentiment_model_lgc[aspect].predict(padded_sentiment_val[aspect])
    pred_sentiment_test = sentiment_model_lgc[aspect].predict(padded_sentiment_test[aspect])
    
    df_sentiment_train_pred[aspect] = np.argmax(pred_sentiment_train, axis=1)
    df_sentiment_train_true_[aspect] = np.argmax(label_sentiment_train[aspect], axis = 1)
    
    df_sentiment_val_pred[aspect] = np.argmax(pred_sentiment_val, axis=1)
    df_sentiment_val_true_[aspect] = np.argmax(label_sentiment_val[aspect], axis = 1)
    
    df_sentiment_test_pred[aspect] = np.argmax(pred_sentiment_test, axis=1)
    df_sentiment_test_true_[aspect] = np.argmax(label_sentiment_test[aspect], axis=1)

15/15 [==============================] - 0s 25ms/step


In [132]:
def get_pd_report_sentiment(df_sentiment_test_true_, df_sentiment_test_pred):
    sentiment_result_bilstm = {}
    for aspect in lst_aspect:
        sentiment_result_bilstm[aspect] = pd.DataFrame(classification_report(df_sentiment_test_true_[aspect].tolist(), df_sentiment_test_pred[aspect].tolist(), output_dict=True)).T
        sentiment_result_bilstm[aspect]['aspect'] = aspect

    output_sentiment_report = pd.DataFrame()
    for indx, val in sentiment_result_bilstm.items():
        output_sentiment_report = pd.concat([output_sentiment_report, sentiment_result_bilstm[indx]])

    return output_aspect_report

In [133]:
sentiment_result_bilstm = {}
# for aspect in lst_aspect:
for aspect in lst_aspect:
    sentiment_result_bilstm[aspect] = pd.DataFrame(classification_report(df_sentiment_test_true_[aspect].tolist(), df_sentiment_test_pred[aspect].tolist(), output_dict=True)).T
    sentiment_result_bilstm[aspect]['aspect'] = aspect

output_sentiment_report = pd.DataFrame()
for indx, val in sentiment_result_bilstm.items():
    output_sentiment_report = pd.concat([output_sentiment_report, sentiment_result_bilstm[indx]])

/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classif

In [134]:
output_sentiment_report

,precision,recall,f1-score,support,aspect
0,0.000000,0.000000,0.000000,3.000000,price
1,0.904382,0.919028,0.911647,247.000000,price
2,0.777778,0.769231,0.773481,91.000000,price
accuracy,0.870968,0.870968,0.870968,0.870968,price
macro avg,0.560720,0.562753,0.561709,341.000000,price
weighted avg,0.862640,0.870968,0.866755,341.000000,price
0,0.822695,0.935484,0.875472,124.000000,shipping
1,0.965517,0.969035,0.967273,549.000000,shipping
2,0.153846,0.062500,0.088889,32.000000,shipping
accuracy,0.921986,0.921986,0.921986,0.921986,shipping


In [135]:
for aspect in lst_aspect:
    print(aspect)
    print(classification_report(df_sentiment_test_true_[aspect].tolist(), df_sentiment_test_pred[aspect].tolist()))

price
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         3
           1       0.90      0.92      0.91       247
           2       0.78      0.77      0.77        91

    accuracy                           0.87       341
   macro avg       0.56      0.56      0.56       341
weighted avg       0.86      0.87      0.87       341

shipping
              precision    recall  f1-score   support

           0       0.82      0.94      0.88       124
           1       0.97      0.97      0.97       549
           2       0.15      0.06      0.09        32

    accuracy                           0.92       705
   macro avg       0.65      0.66      0.64       705
weighted avg       0.90      0.92      0.91       705

outlook
              precision    recall  f1-score   support

           0       0.84      0.65      0.73        95
           1       0.96      0.97      0.97      1118
           2       0.28      0.34      0.31        5

/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classif

In [136]:
for i in lst_aspect:
    with open(f'{path}elmo_bilstm_bigru_con_test_result.txt', "a") as dest:
        dest.write("Classification report for aspect: {} \n".format(i.upper()))
        dest.write(classification_report(df_sentiment_test_true_[i].tolist(), df_sentiment_test_pred[i].tolist()))

/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classif

In [137]:
for i in lst_aspect:
    with open(f'{path}elmo_bilstm_bigru_con_val_result.txt', "a") as dest:
        dest.write("Classification report for aspect: {} \n".format(i.upper()))
        dest.write(classification_report(df_sentiment_val_true_[i].tolist(), df_sentiment_val_pred[i].tolist()))

/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/conda/lib/python3.10/site-packages/sklearn/metrics/_classif

# VI. Overall Results

## 1. BiLSTM

In [138]:
pred_test = aspect_model_bil.predict(padded_aspect_test)
df_pred = round(pd.DataFrame(pred_test), 0)
df_true = label_aspect_test
df_pred.columns = df_true.columns

74/74 [==============================] - 1s 13ms/step


In [139]:
y_pred = []
y_true = []
for i in range(len(df_true)):
    y_pred.append(list(df_pred.loc[i]))
    y_true.append(list(df_true.loc[i]))

In [140]:
list_aspect = df_train.columns[1:9]
list_aspect

Index(['Price', 'Shipping', 'Outlook', 'Quality', 'Size', 'Shop_Service',
       'General', 'Others'],
      dtype='object')

In [141]:
y_pred = []
y_true = []
for i in range(len(df_true)):
    y_pred.append(list(df_pred.loc[i]))
    y_true.append(list(df_true.loc[i]))

In [142]:
aspect_test = []
aspect_pred = []

for row_test, row_pred in zip(y_true, y_pred):
    for index, (col_test, col_pred) in enumerate(zip(list(row_test), list(row_pred))):
        aspect_test.append(bool(col_test) * list_aspect[index])
        aspect_pred.append(bool(col_pred) * list_aspect[index])

## a) Aspect Detection

In [143]:
aspect_report = classification_report(aspect_test, aspect_pred, digits=4, zero_division=1, output_dict=True)
print(classification_report(aspect_test, aspect_pred, digits=4, zero_division=1))

              precision    recall  f1-score   support

                 0.9397    0.9561    0.9478     14062
     General     0.6476    0.6367    0.6421       479
      Others     0.8395    0.7196    0.7749       189
     Outlook     0.9359    0.8954    0.9152      1271
       Price     0.9046    0.6950    0.7861       341
     Quality     0.7878    0.8120    0.7997       686
    Shipping     0.9429    0.9362    0.9395       705
Shop_Service     0.8757    0.7517    0.8090       600
        Size     0.8206    0.8036    0.8120       387

    accuracy                         0.9209     18720
   macro avg     0.8549    0.8007    0.8251     18720
weighted avg     0.9203    0.9209    0.9201     18720



## b) Polarity Detection

In [144]:
y_sen_pred = []
y_sen_true = []

for aspect in lst_aspect:
    pred_sentiment_test = sentiment_model_bil[aspect].predict(padded_sentiment_test[aspect])
    y_sen_pred.extend(list(np.argmax(pred_sentiment_test, axis=1)))
    y_sen_true.extend(list(np.argmax(label_sentiment_test[aspect], axis=1)))

15/15 [==============================] - 0s 13ms/step


In [145]:
replacements = {0: 'negative', 1: 'positive', 2: 'neutral'}

In [146]:
target_names = list(map(str, replacements.values()))
target_names

['negative', 'positive', 'neutral']

In [147]:
polarity_report = classification_report(y_sen_true, y_sen_pred, digits=4, output_dict=True)
print(classification_report(y_sen_true, y_sen_pred, target_names=target_names, digits=4))

              precision    recall  f1-score   support

    negative     0.7043    0.7433    0.7233       596
    positive     0.9183    0.9481    0.9330      3273
     neutral     0.6536    0.5017    0.5676       598

    accuracy                         0.8610      4467
   macro avg     0.7587    0.7310    0.7413      4467
weighted avg     0.8543    0.8610    0.8561      4467



## c) Aspect + Polarity

In [148]:
aspect_polarity_test = []
aspect_polarity_pred = []

for aspect in lst_aspect:
    pred_sentiment_test = sentiment_model_bil[aspect].predict(padded_sentiment_test[aspect])
    pred_sentiment_test = list(np.argmax(pred_sentiment_test, axis=1))
    sen_test = list(np.argmax(label_sentiment_test[aspect], axis=1))
    for i in range(len(sen_test)):
        aspect_polarity_pred.append(f'{aspect},{replacements[pred_sentiment_test[i]]}')
        aspect_polarity_test.append(f'{aspect},{replacements[sen_test[i]]}')

15/15 [==============================] - 0s 13ms/step


In [149]:
aspect_polarity_report = classification_report(aspect_polarity_test, aspect_polarity_pred, digits=4, zero_division=1, output_dict=True)
print(classification_report(aspect_polarity_test, aspect_polarity_pred, digits=4, zero_division=1))

                      precision    recall  f1-score   support

    general,negative     1.0000    0.0000    0.0000        11
     general,neutral     0.7840    0.6978    0.7384       182
    general,positive     0.8101    0.8982    0.8519       285
    outlook,negative     0.7571    0.5579    0.6424        95
     outlook,neutral     0.4400    0.1897    0.2651        58
    outlook,positive     0.9396    0.9884    0.9634      1118
      price,negative     1.0000    0.0000    0.0000         3
       price,neutral     0.7315    0.8681    0.7940        91
      price,positive     0.9442    0.8907    0.9167       247
    quality,negative     0.7500    0.5816    0.6552        98
     quality,neutral     0.5806    0.4954    0.5347       109
    quality,positive     0.8760    0.9456    0.9095       478
   shipping,negative     0.8346    0.8952    0.8638       124
    shipping,neutral     0.0909    0.0625    0.0741        32
   shipping,positive     0.9618    0.9636    0.9627       549
shopser

## d) Overall

In [151]:
aspect_dict = aspect_report['macro avg']
aspect_dict['accuracy'] = aspect_report['accuracy']

polarity_dict  = polarity_report['macro avg']
polarity_dict['accuracy'] = polarity_report['accuracy']

aspect_polarity_dict = aspect_polarity_report['macro avg']
aspect_polarity_dict['accuracy'] = aspect_polarity_report['accuracy']

In [152]:
df_report_bi = pd.DataFrame.from_dict([aspect_dict, polarity_dict, aspect_polarity_dict])
df_report_bi.index = ['Aspect Detection', 'Polarity Detection', 'Aspect + Polarity']
df_report_bi = df_report_bi.drop('support', axis=1)

## 2. BiGRU

In [153]:
pred_test = aspect_model_gru.predict(padded_aspect_test)
df_pred = round(pd.DataFrame(pred_test), 0)
df_true = label_aspect_test
df_pred.columns = df_true.columns

74/74 [==============================] - 1s 13ms/step


In [154]:
y_pred = []
y_true = []
for i in range(len(df_true)):
    y_pred.append(list(df_pred.loc[i]))
    y_true.append(list(df_true.loc[i]))

In [155]:
list_aspect = df_train.columns[1:9]
list_aspect

Index(['Price', 'Shipping', 'Outlook', 'Quality', 'Size', 'Shop_Service',
       'General', 'Others'],
      dtype='object')

In [156]:
y_pred = []
y_true = []
for i in range(len(df_true)):
    y_pred.append(list(df_pred.loc[i]))
    y_true.append(list(df_true.loc[i]))

In [157]:
aspect_test = []
aspect_pred = []

for row_test, row_pred in zip(y_true, y_pred):
    for index, (col_test, col_pred) in enumerate(zip(list(row_test), list(row_pred))):
        aspect_test.append(bool(col_test) * list_aspect[index])
        aspect_pred.append(bool(col_pred) * list_aspect[index])

## a) Aspect Detection

In [158]:
aspect_report = classification_report(aspect_test, aspect_pred, digits=4, zero_division=1, output_dict=True)
print(classification_report(aspect_test, aspect_pred, digits=4, zero_division=1))

              precision    recall  f1-score   support

                 0.9453    0.9602    0.9527     14062
     General     0.7444    0.5595    0.6389       479
      Others     0.8161    0.7513    0.7824       189
     Outlook     0.9302    0.9221    0.9261      1271
       Price     0.8836    0.7566    0.8152       341
     Quality     0.8213    0.8105    0.8158       686
    Shipping     0.9435    0.9475    0.9455       705
Shop_Service     0.8590    0.7917    0.8239       600
        Size     0.8180    0.8708    0.8436       387

    accuracy                         0.9283     18720
   macro avg     0.8624    0.8189    0.8382     18720
weighted avg     0.9267    0.9283    0.9269     18720



## b) Polarity Detection

In [214]:
y_sen_pred = []
y_sen_true = []

for aspect in lst_aspect:
    pred_sentiment_test = sentiment_model_gru[aspect].predict(padded_sentiment_test[aspect])
    y_sen_pred.extend(list(np.argmax(pred_sentiment_test, axis=1)))
    y_sen_true.extend(list(np.argmax(label_sentiment_test[aspect], axis=1)))

15/15 [==============================] - 0s 11ms/step


In [215]:
replacements = {0: 'negative', 1: 'positive', 2: 'neutral'}

In [216]:
target_names = list(map(str, replacements.values()))
target_names

['negative', 'positive', 'neutral']

In [217]:
polarity_report = classification_report(y_sen_true, y_sen_pred, digits=4, output_dict=True)
print(classification_report(y_sen_true, y_sen_pred, target_names=target_names, digits=4))

              precision    recall  f1-score   support

    negative     0.7268    0.7500    0.7382       596
    positive     0.9159    0.9514    0.9333      3273
     neutral     0.6283    0.4749    0.5410       598

    accuracy                         0.8608      4467
   macro avg     0.7570    0.7254    0.7375      4467
weighted avg     0.8522    0.8608    0.8548      4467



## c) Aspect + Polarity

In [218]:
aspect_polarity_test = []
aspect_polarity_pred = []

for aspect in lst_aspect:
    pred_sentiment_test = sentiment_model_gru[aspect].predict(padded_sentiment_test[aspect])
    pred_sentiment_test = list(np.argmax(pred_sentiment_test, axis=1))
    sen_test = list(np.argmax(label_sentiment_test[aspect], axis=1))
    for i in range(len(sen_test)):
        aspect_polarity_pred.append(f'{aspect},{replacements[pred_sentiment_test[i]]}')
        aspect_polarity_test.append(f'{aspect},{replacements[sen_test[i]]}')

15/15 [==============================] - 0s 11ms/step


In [219]:
aspect_polarity_report = classification_report(aspect_polarity_test, aspect_polarity_pred, digits=4, zero_division=1, output_dict=True)
print(classification_report(aspect_polarity_test, aspect_polarity_pred, digits=4, zero_division=1))

                      precision    recall  f1-score   support

    general,negative     1.0000    0.0000    0.0000        11
     general,neutral     0.7843    0.6593    0.7164       182
    general,positive     0.7908    0.9018    0.8426       285
    outlook,negative     0.6981    0.7789    0.7363        95
     outlook,neutral     0.2963    0.2759    0.2857        58
    outlook,positive     0.9703    0.9642    0.9672      1118
      price,negative     1.0000    0.0000    0.0000         3
       price,neutral     0.7692    0.7692    0.7692        91
      price,positive     0.9040    0.9150    0.9095       247
    quality,negative     0.6500    0.5306    0.5843        98
     quality,neutral     0.5000    0.3303    0.3978       109
    quality,positive     0.8537    0.9519    0.9001       478
   shipping,negative     0.7959    0.9435    0.8635       124
    shipping,neutral     0.0000    0.0000    0.0000        32
   shipping,positive     0.9586    0.9709    0.9647       549
shopser

## d) Overall

In [220]:
aspect_dict = aspect_report['macro avg']
aspect_dict['accuracy'] = aspect_report['accuracy']

polarity_dict  = polarity_report['macro avg']
polarity_dict['accuracy'] = polarity_report['accuracy']

aspect_polarity_dict = aspect_polarity_report['macro avg']
aspect_polarity_dict['accuracy'] = aspect_polarity_report['accuracy']

In [221]:
df_report_gru = pd.DataFrame.from_dict([aspect_dict, polarity_dict, aspect_polarity_dict])
df_report_gru.index = ['Aspect Detection', 'Polarity Detection', 'Aspect + Polarity']
df_report_gru = df_report_gru.drop('support', axis=1)

## 3. BiLSTM + Conv1D

In [168]:
pred_test = aspect_model_bil_con.predict(padded_aspect_test)
df_pred = round(pd.DataFrame(pred_test), 0)
df_true = label_aspect_test
df_pred.columns = df_true.columns

74/74 [==============================] - 1s 16ms/step


In [169]:
y_pred = []
y_true = []
for i in range(len(df_true)):
    y_pred.append(list(df_pred.loc[i]))
    y_true.append(list(df_true.loc[i]))

In [170]:
list_aspect = df_train.columns[1:9]
list_aspect

Index(['Price', 'Shipping', 'Outlook', 'Quality', 'Size', 'Shop_Service',
       'General', 'Others'],
      dtype='object')

In [171]:
y_pred = []
y_true = []
for i in range(len(df_true)):
    y_pred.append(list(df_pred.loc[i]))
    y_true.append(list(df_true.loc[i]))

In [172]:
aspect_test = []
aspect_pred = []

for row_test, row_pred in zip(y_true, y_pred):
    for index, (col_test, col_pred) in enumerate(zip(list(row_test), list(row_pred))):
        aspect_test.append(bool(col_test) * list_aspect[index])
        aspect_pred.append(bool(col_pred) * list_aspect[index])

## a) Aspect Detection

In [173]:
aspect_report = classification_report(aspect_test, aspect_pred, digits=4, zero_division=1, output_dict=True)
print(classification_report(aspect_test, aspect_pred, digits=4, zero_division=1))

              precision    recall  f1-score   support

                 0.9488    0.9597    0.9543     14062
     General     0.7214    0.6054    0.6583       479
      Others     0.8750    0.6667    0.7568       189
     Outlook     0.8985    0.9607    0.9285      1271
       Price     0.8854    0.7478    0.8108       341
     Quality     0.8333    0.7945    0.8134       686
    Shipping     0.9415    0.9589    0.9501       705
Shop_Service     0.8926    0.8033    0.8456       600
        Size     0.8568    0.8656    0.8612       387

    accuracy                         0.9309     18720
   macro avg     0.8726    0.8181    0.8421     18720
weighted avg     0.9295    0.9309    0.9296     18720



## b) Polarity Detection

In [222]:
y_sen_pred = []
y_sen_true = []

for aspect in lst_aspect:
    pred_sentiment_test = sentiment_model_bil_con[aspect].predict(padded_sentiment_test[aspect])
    y_sen_pred.extend(list(np.argmax(pred_sentiment_test, axis=1)))
    y_sen_true.extend(list(np.argmax(label_sentiment_test[aspect], axis=1)))

15/15 [==============================] - 0s 13ms/step


In [223]:
replacements = {0: 'negative', 1: 'positive', 2: 'neutral'}

In [224]:
target_names = list(map(str, replacements.values()))
target_names

['negative', 'positive', 'neutral']

In [226]:
polarity_report = classification_report(y_sen_true, y_sen_pred, digits=4, output_dict=True)
print(classification_report(y_sen_true, y_sen_pred, target_names=target_names, digits=4))

              precision    recall  f1-score   support

    negative     0.7305    0.7869    0.7577       596
    positive     0.9331    0.9496    0.9412      3273
     neutral     0.6397    0.5284    0.5788       598

    accuracy                         0.8715      4467
   macro avg     0.7678    0.7550    0.7592      4467
weighted avg     0.8668    0.8715    0.8682      4467



## c) Aspect + Polarity

In [227]:
aspect_polarity_test = []
aspect_polarity_pred = []

for aspect in lst_aspect:
    pred_sentiment_test = sentiment_model_bil_con[aspect].predict(padded_sentiment_test[aspect])
    pred_sentiment_test = list(np.argmax(pred_sentiment_test, axis=1))
    sen_test = list(np.argmax(label_sentiment_test[aspect], axis=1))
    for i in range(len(sen_test)):
        aspect_polarity_pred.append(f'{aspect},{replacements[pred_sentiment_test[i]]}')
        aspect_polarity_test.append(f'{aspect},{replacements[sen_test[i]]}')

15/15 [==============================] - 0s 13ms/step


In [228]:
aspect_polarity_report = classification_report(aspect_polarity_test, aspect_polarity_pred, digits=4, zero_division=1, output_dict=True)
print(classification_report(aspect_polarity_test, aspect_polarity_pred, digits=4, zero_division=1))

                      precision    recall  f1-score   support

    general,negative     1.0000    0.0000    0.0000        11
     general,neutral     0.8141    0.6978    0.7515       182
    general,positive     0.8168    0.9228    0.8666       285
    outlook,negative     0.7374    0.7684    0.7526        95
     outlook,neutral     0.3800    0.3276    0.3519        58
    outlook,positive     0.9661    0.9696    0.9679      1118
      price,negative     1.0000    0.0000    0.0000         3
       price,neutral     0.7087    0.8022    0.7526        91
      price,positive     0.9202    0.8866    0.9031       247
    quality,negative     0.7375    0.6020    0.6629        98
     quality,neutral     0.4769    0.5688    0.5188       109
    quality,positive     0.9200    0.9142    0.9171       478
   shipping,negative     0.8085    0.9194    0.8604       124
    shipping,neutral     0.2000    0.0312    0.0541        32
   shipping,positive     0.9589    0.9763    0.9675       549
shopser

## d) Overall

In [229]:
aspect_dict = aspect_report['macro avg']
aspect_dict['accuracy'] = aspect_report['accuracy']

polarity_dict  = polarity_report['macro avg']
polarity_dict['accuracy'] = polarity_report['accuracy']

aspect_polarity_dict = aspect_polarity_report['macro avg']
aspect_polarity_dict['accuracy'] = aspect_polarity_report['accuracy']

In [230]:
df_report_bil_con = pd.DataFrame.from_dict([aspect_dict, polarity_dict, aspect_polarity_dict])
df_report_bil_con.index = ['Aspect Detection', 'Polarity Detection', 'Aspect + Polarity']
df_report_bil_con = df_report_bil_con.drop('support', axis=1)

## 4. BiGRU + Conv1D

In [231]:
pred_test = aspect_model_big_con.predict(padded_aspect_test)
df_pred = round(pd.DataFrame(pred_test), 0)
df_true = label_aspect_test
df_pred.columns = df_true.columns

74/74 [==============================] - 1s 13ms/step


In [232]:
y_pred = []
y_true = []
for i in range(len(df_true)):
    y_pred.append(list(df_pred.loc[i]))
    y_true.append(list(df_true.loc[i]))

In [233]:
list_aspect = df_train.columns[1:9]
list_aspect

Index(['Price', 'Shipping', 'Outlook', 'Quality', 'Size', 'Shop_Service',
       'General', 'Others'],
      dtype='object')

In [234]:
y_pred = []
y_true = []
for i in range(len(df_true)):
    y_pred.append(list(df_pred.loc[i]))
    y_true.append(list(df_true.loc[i]))

In [235]:
aspect_test = []
aspect_pred = []

for row_test, row_pred in zip(y_true, y_pred):
    for index, (col_test, col_pred) in enumerate(zip(list(row_test), list(row_pred))):
        aspect_test.append(bool(col_test) * list_aspect[index])
        aspect_pred.append(bool(col_pred) * list_aspect[index])

## a) Aspect Detection

In [236]:
aspect_report = classification_report(aspect_test, aspect_pred, digits=4, zero_division=1, output_dict=True)
print(classification_report(aspect_test, aspect_pred, digits=4, zero_division=1))

              precision    recall  f1-score   support

                 0.9499    0.9610    0.9554     14062
     General     0.7453    0.5804    0.6526       479
      Others     0.8503    0.7513    0.7978       189
     Outlook     0.9440    0.9158    0.9297      1271
       Price     0.8914    0.7947    0.8403       341
     Quality     0.8088    0.8265    0.8176       686
    Shipping     0.9322    0.9560    0.9440       705
Shop_Service     0.8761    0.8367    0.8559       600
        Size     0.8262    0.8966    0.8600       387

    accuracy                         0.9326     18720
   macro avg     0.8694    0.8355    0.8504     18720
weighted avg     0.9314    0.9326    0.9316     18720



## b) Polarity Detection

In [237]:
y_sen_pred = []
y_sen_true = []

for aspect in lst_aspect:
    pred_sentiment_test = sentiment_model_big_con[aspect].predict(padded_sentiment_test[aspect])
    y_sen_pred.extend(list(np.argmax(pred_sentiment_test, axis=1)))
    y_sen_true.extend(list(np.argmax(label_sentiment_test[aspect], axis=1)))

15/15 [==============================] - 0s 12ms/step


In [238]:
replacements = {0: 'negative', 1: 'positive', 2: 'neutral'}

In [239]:
target_names = list(map(str, replacements.values()))
target_names

['negative', 'positive', 'neutral']

In [240]:
polarity_report = classification_report(y_sen_true, y_sen_pred, digits=4, output_dict=True)
print(classification_report(y_sen_true, y_sen_pred, target_names=target_names, digits=4))

              precision    recall  f1-score   support

    negative     0.7600    0.7970    0.7781       596
    positive     0.9378    0.9493    0.9435      3273
     neutral     0.6371    0.5635    0.5980       598

    accuracy                         0.8773      4467
   macro avg     0.7783    0.7699    0.7732      4467
weighted avg     0.8738    0.8773    0.8752      4467



## c) Aspect + Polarity

In [241]:
aspect_polarity_test = []
aspect_polarity_pred = []

for aspect in lst_aspect:
    pred_sentiment_test = sentiment_model_big_con[aspect].predict(padded_sentiment_test[aspect])
    pred_sentiment_test = list(np.argmax(pred_sentiment_test, axis=1))
    sen_test = list(np.argmax(label_sentiment_test[aspect], axis=1))
    for i in range(len(sen_test)):
        aspect_polarity_pred.append(f'{aspect},{replacements[pred_sentiment_test[i]]}')
        aspect_polarity_test.append(f'{aspect},{replacements[sen_test[i]]}')

15/15 [==============================] - 0s 12ms/step


In [242]:
aspect_polarity_report = classification_report(aspect_polarity_test, aspect_polarity_pred, digits=4, zero_division=1, output_dict=True)
print(classification_report(aspect_polarity_test, aspect_polarity_pred, digits=4, zero_division=1))

                      precision    recall  f1-score   support

    general,negative     1.0000    0.0000    0.0000        11
     general,neutral     0.7831    0.7143    0.7471       182
    general,positive     0.8173    0.8947    0.8543       285
    outlook,negative     0.7451    0.8000    0.7716        95
     outlook,neutral     0.3393    0.3276    0.3333        58
    outlook,positive     0.9695    0.9651    0.9673      1118
      price,negative     1.0000    0.0000    0.0000         3
       price,neutral     0.7576    0.8242    0.7895        91
      price,positive     0.9256    0.9069    0.9162       247
    quality,negative     0.7614    0.6837    0.7204        98
     quality,neutral     0.5167    0.5688    0.5415       109
    quality,positive     0.9224    0.9205    0.9215       478
   shipping,negative     0.8310    0.9516    0.8872       124
    shipping,neutral     0.0000    0.0000    0.0000        32
   shipping,positive     0.9625    0.9818    0.9720       549
shopser

## d) Overall

In [243]:
aspect_dict = aspect_report['macro avg']
aspect_dict['accuracy'] = aspect_report['accuracy']

polarity_dict  = polarity_report['macro avg']
polarity_dict['accuracy'] = polarity_report['accuracy']

aspect_polarity_dict = aspect_polarity_report['macro avg']
aspect_polarity_dict['accuracy'] = aspect_polarity_report['accuracy']

In [244]:
df_report_big_con = pd.DataFrame.from_dict([aspect_dict, polarity_dict, aspect_polarity_dict])
df_report_big_con.index = ['Aspect Detection', 'Polarity Detection', 'Aspect + Polarity']
df_report_big_con = df_report_big_con.drop('support', axis=1)

## 5. BiLSTM + BiGRU + Conv1D

In [245]:
pred_test = aspect_model_lgc.predict(padded_aspect_test)
df_pred = round(pd.DataFrame(pred_test), 0)
df_true = label_aspect_test
df_pred.columns = df_true.columns

74/74 [==============================] - 2s 27ms/step


In [246]:
y_pred = []
y_true = []
for i in range(len(df_true)):
    y_pred.append(list(df_pred.loc[i]))
    y_true.append(list(df_true.loc[i]))

In [247]:
list_aspect = df_train.columns[1:9]
list_aspect

Index(['Price', 'Shipping', 'Outlook', 'Quality', 'Size', 'Shop_Service',
       'General', 'Others'],
      dtype='object')

In [248]:
y_pred = []
y_true = []
for i in range(len(df_true)):
    y_pred.append(list(df_pred.loc[i]))
    y_true.append(list(df_true.loc[i]))

In [249]:
aspect_test = []
aspect_pred = []

for row_test, row_pred in zip(y_true, y_pred):
    for index, (col_test, col_pred) in enumerate(zip(list(row_test), list(row_pred))):
        aspect_test.append(bool(col_test) * list_aspect[index])
        aspect_pred.append(bool(col_pred) * list_aspect[index])

## a) Aspect Detection

In [250]:
aspect_report = classification_report(aspect_test, aspect_pred, digits=4, zero_division=1, output_dict=True)
print(classification_report(aspect_test, aspect_pred, digits=4, zero_division=1))

              precision    recall  f1-score   support

                 0.9455    0.9628    0.9541     14062
     General     0.7435    0.5992    0.6636       479
      Others     0.8249    0.7725    0.7978       189
     Outlook     0.9319    0.9158    0.9238      1271
       Price     0.8972    0.7419    0.8122       341
     Quality     0.8216    0.7988    0.8101       686
    Shipping     0.9413    0.9560    0.9486       705
Shop_Service     0.8989    0.7850    0.8381       600
        Size     0.8375    0.8656    0.8513       387

    accuracy                         0.9304     18720
   macro avg     0.8714    0.8220    0.8444     18720
weighted avg     0.9289    0.9304    0.9291     18720



## b) Polarity Detection

In [251]:
y_sen_pred = []
y_sen_true = []

for aspect in lst_aspect:
    pred_sentiment_test = sentiment_model_lgc[aspect].predict(padded_sentiment_test[aspect])
    y_sen_pred.extend(list(np.argmax(pred_sentiment_test, axis=1)))
    y_sen_true.extend(list(np.argmax(label_sentiment_test[aspect], axis=1)))

15/15 [==============================] - 0s 25ms/step


In [252]:
replacements = {0: 'negative', 1: 'positive', 2: 'neutral'}

In [253]:
target_names = list(map(str, replacements.values()))
target_names

['negative', 'positive', 'neutral']

In [254]:
polarity_report = classification_report(y_sen_true, y_sen_pred, digits=4, output_dict=True)
print(classification_report(y_sen_true, y_sen_pred, target_names=target_names, digits=4))

              precision    recall  f1-score   support

    negative     0.7747    0.7617    0.7682       596
    positive     0.9296    0.9484    0.9389      3273
     neutral     0.6125    0.5552    0.5825       598

    accuracy                         0.8708      4467
   macro avg     0.7723    0.7551    0.7632      4467
weighted avg     0.8665    0.8708    0.8684      4467



## c) Aspect + Polarity

In [255]:
aspect_polarity_test = []
aspect_polarity_pred = []

for aspect in lst_aspect:
    pred_sentiment_test = sentiment_model_lgc[aspect].predict(padded_sentiment_test[aspect])
    pred_sentiment_test = list(np.argmax(pred_sentiment_test, axis=1))
    sen_test = list(np.argmax(label_sentiment_test[aspect], axis=1))
    for i in range(len(sen_test)):
        aspect_polarity_pred.append(f'{aspect},{replacements[pred_sentiment_test[i]]}')
        aspect_polarity_test.append(f'{aspect},{replacements[sen_test[i]]}')

15/15 [==============================] - 0s 25ms/step


In [256]:
aspect_polarity_report = classification_report(aspect_polarity_test, aspect_polarity_pred, digits=4, zero_division=1, output_dict=True)
print(classification_report(aspect_polarity_test, aspect_polarity_pred, digits=4, zero_division=1))

                      precision    recall  f1-score   support

    general,negative     1.0000    0.0000    0.0000        11
     general,neutral     0.7937    0.6978    0.7427       182
    general,positive     0.8082    0.9018    0.8524       285
    outlook,negative     0.8378    0.6526    0.7337        95
     outlook,neutral     0.2778    0.3448    0.3077        58
    outlook,positive     0.9636    0.9696    0.9666      1118
      price,negative     1.0000    0.0000    0.0000         3
       price,neutral     0.7778    0.7692    0.7735        91
      price,positive     0.9044    0.9190    0.9116       247
    quality,negative     0.7529    0.6531    0.6995        98
     quality,neutral     0.4797    0.5413    0.5086       109
    quality,positive     0.9161    0.9142    0.9152       478
   shipping,negative     0.8227    0.9355    0.8755       124
    shipping,neutral     0.1538    0.0625    0.0889        32
   shipping,positive     0.9655    0.9690    0.9673       549
shopser

## d) Overall

In [257]:
aspect_dict = aspect_report['macro avg']
aspect_dict['accuracy'] = aspect_report['accuracy']

polarity_dict  = polarity_report['macro avg']
polarity_dict['accuracy'] = polarity_report['accuracy']

aspect_polarity_dict = aspect_polarity_report['macro avg']
aspect_polarity_dict['accuracy'] = aspect_polarity_report['accuracy']

In [258]:
df_report_lgc = pd.DataFrame.from_dict([aspect_dict, polarity_dict, aspect_polarity_dict])
df_report_lgc.index = ['Aspect Detection', 'Polarity Detection', 'Aspect + Polarity']
df_report_lgc = df_report_lgc.drop('support', axis=1)

In [259]:
res_all = [df_report_bi, df_report_gru, df_report_bil_con, df_report_big_con, df_report_lgc]
model_names = ['BiLSTM', 'BiGRU', 'BiLSTM + Conv1D', 'BiGRU + Conv1D', 'BiLSTM + BiGRU + Conv1D']
for i in range(len(res_all)):
    print(model_names[i])
    print(res_all[i])
    print('-' * 60)

BiLSTM
                    precision    recall  f1-score  accuracy
Aspect Detection     0.854910  0.800691  0.825143  0.920940
Polarity Detection   0.758735  0.731007  0.741287  0.860981
Aspect + Polarity    0.768824  0.609640  0.609739  0.860981
------------------------------------------------------------
BiGRU
                    precision    recall  f1-score  accuracy
Aspect Detection     0.871367  0.821967  0.844399  0.930395
Polarity Detection   0.757010  0.725446  0.737500  0.860757
Aspect + Polarity    0.747793  0.611730  0.605705  0.860757
------------------------------------------------------------
BiLSTM + Conv1D
                    precision    recall  f1-score  accuracy
Aspect Detection     0.871367  0.821967  0.844399  0.930395
Polarity Detection   0.767753  0.754976  0.759225  0.871502
Aspect + Polarity    0.756517  0.635358  0.630628  0.871502
------------------------------------------------------------
BiGRU + Conv1D
                    precision    recall  f1-score  ac

In [260]:
import zipfile
import os
from IPython.display import FileLink

def zip_dir(directory = os.curdir, file_name = 'directory.zip'):
    """
    zip all the files in a directory
    
    Parameters
    _____
    directory: str
        directory needs to be zipped, defualt is current working directory
        
    file_name: str
        the name of the zipped file (including .zip), default is 'directory.zip'
        
    Returns
    _____
    Creates a hyperlink, which can be used to download the zip file)
    """
    os.chdir(directory)
    zip_ref = zipfile.ZipFile(file_name, mode='w')
    for folder, _, files in os.walk(directory):
        for file in files:
            if file_name in file:
                pass
            else:
                zip_ref.write(os.path.join(folder, file))

    return FileLink(file_name)

In [261]:
zip_dir()

/kaggle/working/directory.zip